<a href="https://colab.research.google.com/github/patrickallanfaustino/patrickallanfaustino.github.io/blob/main/aula_aprendizagem_emmsb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aprendizado de máquina aplicado à bioinformática
## Classificar inibidores de um alvo proteico a partir do ChEMBL

Aula prática. A partir de dados reais de inibidores da **acetilcolinesterase**
(AChE) extraídos do ChEMBL, vamos treinar modelos que recebem qualquer molécula
(pelo seu SMILES) e a classificam como **FORTE** ou **FRACO** para esse alvo — ou
se **abstêm** (classe **INDEFINIDA**) quando não têm base para decidir.

A acetilcolinesterase degrada o neurotransmissor acetilcolina. Seus inibidores
incluem fármacos contra o Alzheimer (donepezila, rivastigmina), inseticidas
organofosforados e agentes neurotóxicos — um alvo de grande relevância
biomédica e toxicológica, com milhares de moléculas medidas no ChEMBL.

**Como usar este notebook.** Cada célula de código vem depois de uma célula de
texto que explica *o que* ela faz, *por que* é necessária e *o que observar* no
resultado. Rode as células em ordem, de cima para baixo. O notebook inteiro roda
em CPU, sem GPU, em poucos minutos.

**A decisão de projeto central: treino binário, resposta com abstenção.** O modelo
é treinado só para separar FORTE de FRACO (o rótulo, por um limiar de pIC50). Mas,
na hora de responder, ele pode **se abster** e devolver a classe **INDEFINIDA** por
duas razões distintas: (a) **abstenção por ambiguidade** — a probabilidade fica
perto do meio, sem evidência clara; (b) **fora do domínio de aplicabilidade** — a
molécula não se parece com nada que o modelo viu, então ele não deveria opinar.
Saber quando não decidir é parte do trabalho.

## Seção 0 — Ambiente

Antes de qualquer ciência, deixamos o ambiente reprodutível: instalamos o que
falta, importamos as bibliotecas agrupadas por finalidade, imprimimos as versões
e fixamos **uma única semente** propagada a tudo que é aleatório. Reprodutível
quer dizer que rodar de novo dá o mesmo resultado — condição para poder confiar
em qualquer número que aparecer depois.

### 0.1 — Instalação

`scikit-learn`, `pandas`, `numpy`, `matplotlib` e `torch` já vêm no Colab. Faltam
o `rdkit` (química), o `chembl_webresource_client` (acesso ao ChEMBL) e o
`plotly` (gráficos interativos). Instalamos apenas o ausente, testando o
`import` antes — assim a célula é rápida quando já está tudo instalado.

Este bloco usa `subprocess` em vez da mágica `!pip` para funcionar igual no Colab
e em qualquer outro kernel. Pode levar cerca de um minuto na primeira execução.

In [ ]:
# Célula 01 — Instalação dos pacotes ausentes (rdkit, plotly, ChEMBL)
import importlib.util
import subprocess
import sys

# pares (nome para importar, nome para instalar via pip)
pacotes = [
    ("rdkit", "rdkit"),
    ("plotly", "plotly"),
    ("chembl_webresource_client", "chembl_webresource_client"),
    ("shap", "shap"),
    ("umap", "umap-learn"),
    ("tqdm", "tqdm"),
]

# instala cada um so se ainda nao puder ser importado
for nome_import, nome_pip in pacotes:
    if importlib.util.find_spec(nome_import) is not None:
        print("ja instalado:", nome_import)
    else:
        print("instalando:", nome_pip)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", nome_pip], check=True)
print("instalacao concluida")

### 0.2 — Importações, agrupadas por finalidade

Importar tudo em um lugar só, comentado, evita surpresas mais adiante e mostra ao
leitor o mapa de ferramentas da aula.

In [ ]:
# Célula 02 — Importações agrupadas por finalidade
# --- básicas: dados e números ---
import numpy as np
import pandas as pd

# --- química: ler moléculas, calcular descritores e fingerprints, desenhar ---
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import DataStructs
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")   # silencia avisos verbosos do RDKit

# --- modelos e avaliação (tudo scikit-learn: uma única interface) ---
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import (matthews_corrcoef, roc_auc_score, roc_curve,
                             precision_recall_curve, confusion_matrix,
                             classification_report)

# --- rede neural "aberta" em PyTorch (só na subseção 5b) ---
import torch
import torch.nn as nn

# --- gráficos interativos ---
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- desenho estático e utilidades ---
import matplotlib.pyplot as plt
import time
import joblib
from tqdm.auto import tqdm

print("importacoes ok")

### 0.3 — Versões e semente

Imprimimos as versões (inclusive a do PyTorch, verificada em tempo de execução, e
não suposta) e confirmamos que **não há GPU**: a aula roda em CPU por escolha,
para ser idêntica em qualquer máquina. A semente é uma constante única no topo,
propagada a `random`, `numpy` e `torch`, e passada como `random_state` a todo
estimador daqui para frente.

In [ ]:
# Célula 03 — Versões das bibliotecas e semente reprodutível
import random
import sklearn

SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)
torch.manual_seed(SEMENTE)

print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("torch       ", torch.__version__)
print("cuda disponivel:", torch.cuda.is_available(), "(esperado: False, rodamos em CPU)")
DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("dispositivo :", DISPOSITIVO)

## Seção 1 — A pergunta e os dados

Antes de qualquer código: **o que estamos medindo?**

Um inibidor é uma molécula que reduz a atividade da enzima. Quanto menos inibidor
é preciso para reduzir a atividade pela metade, mais potente ele é. Essa
quantidade é o **IC50** — a concentração que inibe 50% da atividade. IC50 baixo =
inibidor potente.

Como o IC50 varia por ordens de grandeza (de nanomolar a milimolar), trabalhamos
com o **pIC50**:

$$\text{pIC50} = -\log_{10}(\text{IC50 em mol/L})$$

A escala logarítmica transforma "10 vezes mais potente" em "+1 unidade",
tornando a variável comparável e bem-comportada para modelos. pIC50 **alto** =
inibidor **potente**.

**Uma ressalva honesta:** IC50 não é energia de ligação. Ele depende da
concentração de substrato, do $K_m$ da enzima e do formato do ensaio (relação de
Cheng-Prusoff). Medidas de laboratórios e ensaios diferentes têm ruído — motivo
pelo qual a curadoria da Seção 2 existe.

### 1.1 — Carregar os dados

Nós já extraímos os dados do ChEMBL e deixamos o CSV num **link público** no
repositório. Assim o notebook lê direto da internet — **sem precisar subir
arquivo nenhum** no Colab. É o caminho padrão, e é rápido.

In [ ]:
# Célula 04 — Carrega o CSV de dados (URL com fallback local)
URL_DADOS = ("https://raw.githubusercontent.com/monteirotorres/ml/"
             "main/data/dados_alvo_bruto.csv")

# le o CSV direto da URL; se a internet falhar, tenta um arquivo local de mesmo nome
try:
    dados_brutos = pd.read_csv(URL_DADOS)
    print("lidos do repositorio (URL):", len(dados_brutos), "registros")
except Exception as erro:
    print("URL indisponivel (", type(erro).__name__, "); tentando arquivo local")
    dados_brutos = pd.read_csv("dados_alvo_bruto.csv")
    print("lidos do arquivo local:", len(dados_brutos), "registros")

dados_brutos.head()

**Opcional — como o CSV foi produzido.** A célula abaixo re-extrai os dados
direto da API do ChEMBL, para mostrar de onde eles vêm. É **lenta** (alguns
minutos, baixa ~10 mil medidas uma a uma), então vem **desligada** — deixe
`EXECUTAR_EXTRACAO_API = False` em sala e rode em casa se tiver curiosidade. O
alvo é resolvido pelo **nome do gene**, nunca fixado no código: trocar de alvo é
mudar uma linha.

In [ ]:
# Célula 05 — Extração opcional do ChEMBL pela API (desligada por padrão)
ALVO_GENE = "ACHE"                    # gene da acetilcolinesterase humana
EXECUTAR_EXTRACAO_API = False         # mude para True para baixar da API (lento)

if EXECUTAR_EXTRACAO_API:
    from chembl_webresource_client.new_client import new_client

    # 1. resolve o ChEMBL ID do alvo a partir do simbolo do gene
    alvo_id = None
    for alvo in new_client.target.filter(target_synonym__icontains=ALVO_GENE):
        if alvo.get("organism") == "Homo sapiens":
            alvo_id = alvo["target_chembl_id"]
            break
    print("alvo resolvido pela API:", alvo_id)

    # 2. baixa as atividades IC50 desse alvo, uma a uma, para uma tabela
    colunas = ["molecule_chembl_id", "canonical_smiles", "standard_type",
               "standard_relation", "standard_value", "standard_units",
               "pchembl_value", "assay_type", "assay_chembl_id",
               "data_validity_comment", "document_chembl_id", "target_chembl_id"]
    linhas = []
    for atividade in new_client.activity.filter(target_chembl_id=alvo_id,
                                                standard_type="IC50"):
        linha = {}
        for coluna in colunas:
            linha[coluna] = atividade.get(coluna)
        linhas.append(linha)
    dados_brutos = pd.DataFrame(linhas)
    print("re-extraidos da API:", len(dados_brutos), "registros")
else:
    print("extracao da API desligada; usando o CSV lido acima")

### 1.2 — Primeiro olhar

Antes de mexer em qualquer coisa, olhamos o formato bruto: quantas linhas,
quantas moléculas únicas, e a distribuição das relações de medida (`=`, `>`,
`<`). As relações vão importar muito na curadoria — uma medida "> 10000 nM"
significa "pelo menos tão fraco quanto isso", e essa informação não pode ser
jogada fora.

In [ ]:
# Célula 06 — Primeiro olhar: contagens e amostra dos dados brutos
print("linhas totais      :", len(dados_brutos))
print("moleculas unicas   :", dados_brutos["molecule_chembl_id"].nunique())
print("com SMILES         :", dados_brutos["canonical_smiles"].notna().sum())
print()
print("relacoes de medida (standard_relation):")
print(dados_brutos["standard_relation"].value_counts(dropna=False))

## Seção 2 — Curadoria, com tabela de proveniência

Dados de repositório público chegam sujos. Cada filtro abaixo corrige um defeito
específico, e **cada etapa imprime quantas linhas entraram e quantas saíram** —
nenhuma transformação silenciosa. Ao final, montamos uma tabela de proveniência
(etapa, n restante, n removido, motivo) e um gráfico de funil.

Começamos criando um registro da proveniência e uma cópia de trabalho.

In [ ]:
# Célula 07 — Infraestrutura da proveniência (lista + função registrar_etapa)
proveniencia = []   # lista de (etapa, n_restante, n_removido, motivo)

def registrar_etapa(nome, antes, depois, motivo):
    """Imprime o efeito de uma etapa e o guarda para a tabela de proveniencia."""
    removido = antes - depois
    proveniencia.append((nome, depois, removido, motivo))
    print(f"{nome:32s} | antes {antes:5d} -> depois {depois:5d} | removidos {removido:5d}")

curados = dados_brutos.copy()
registrar_etapa("bruto", len(curados), len(curados), "dados como vieram do ChEMBL")

### 2.1 — SMILES ausentes ou inválidos

Um SMILES é a representação textual da molécula. Se estiver faltando ou não puder
ser lido pelo RDKit (parênteses errados, valência impossível), a linha é inútil —
não há molécula para calcular nada. Testamos cada SMILES com um laço explícito.

In [ ]:
# Célula 08 — Curadoria: remove SMILES ausentes ou inválidos
antes = len(curados)
curados = curados[curados["canonical_smiles"].notna()].copy()

smiles_valido = []
for smiles in curados["canonical_smiles"]:
    molecula = Chem.MolFromSmiles(smiles)
    smiles_valido.append(molecula is not None)
curados = curados[pd.Series(smiles_valido, index=curados.index)].copy()

registrar_etapa("SMILES valido", antes, len(curados), "SMILES ausente ou ilegivel pelo RDKit")

### 2.2 — Comentários de validade e tipo de ensaio

O ChEMBL marca medidas suspeitas em `data_validity_comment` ("Outside typical
range", "Potential transcription error"). Removemos as marcadas. Também mantemos
apenas ensaios de ligação/funcionais do tipo `B` (binding), que medem inibição
direta, descartando os demais formatos que não são comparáveis.

In [ ]:
# Célula 09 — Curadoria: filtra validade e tipo de ensaio
antes = len(curados)
curados = curados[curados["data_validity_comment"].isna()].copy()
registrar_etapa("validade ok", antes, len(curados), "data_validity_comment sinalizado")

antes = len(curados)
curados = curados[curados["assay_type"] == "B"].copy()
registrar_etapa("ensaio tipo B", antes, len(curados), "ensaio nao-binding (nao comparavel)")

### 2.3 — Unidades e valores numéricos

Queremos tudo em **nanomolar (nM)** para que o pIC50 faça sentido. Aqui vale um
detalhe do ChEMBL: usamos o campo `standard_units`, que é a unidade **já
padronizada** pela curadoria do ChEMBL. Ou seja, os µM e pM que apareciam nos
artigos originais **já foram convertidos para nM** rio acima — por isso a coluna é
quase toda "nM". Vamos primeiro **olhar** a distribuição de unidades antes de
filtrar (nunca uma etapa silenciosa).

In [ ]:
# Célula 10 — Inspeção das unidades de medida
print("distribuicao de unidades (standard_units):")
print(curados["standard_units"].value_counts(dropna=False))

Mesmo assim, para o código ficar **correto em qualquer alvo** (nem todo
conjunto vem tão limpo), convertemos explicitamente as unidades molares
recuperáveis para nM — µM, mM, M, pM — em vez de descartá-las. Imprimimos quantas
linhas foram convertidas. O que sobra e não é molar (por exemplo `ug.mL-1`,
concentração em massa, que exigiria a massa molecular) ou está malformado será
descartado no filtro seguinte.

In [ ]:
# Célula 11 — Curadoria: converte as unidades para nM
# fatores para converter cada unidade molar para nM
fator_para_nM = {
    "nM": 1.0, "uM": 1000.0, "um": 1000.0, "µM": 1000.0,
    "mM": 1e6, "M": 1e9, "pM": 0.001, "fM": 1e-6,
}
antes = len(curados)
convertidas = 0
valores_nM = []
unidades_nM = []
for valor, unidade in zip(curados["standard_value"], curados["standard_units"]):
    if unidade in fator_para_nM:
        try:
            valores_nM.append(float(valor) * fator_para_nM[unidade])
            unidades_nM.append("nM")
            if unidade != "nM":
                convertidas = convertidas + 1
        except (TypeError, ValueError):
            valores_nM.append(np.nan); unidades_nM.append(unidade)
    else:
        valores_nM.append(np.nan); unidades_nM.append(unidade)   # nao-molar: sera descartado
curados["standard_value"] = valores_nM
curados["standard_units"] = unidades_nM
print("linhas convertidas de outra unidade molar para nM:", convertidas)

# agora filtra o que sobrou: manter nM com valor numerico positivo
curados = curados[curados["standard_units"] == "nM"].copy()
curados = curados[curados["standard_value"].notna()].copy()
curados = curados[curados["standard_value"] > 0].copy()
registrar_etapa("nM, valor > 0", antes, len(curados),
                "unidade nao-molar (ex. ug/mL), malformada, ou valor ausente/<=0")

### 2.4 — Medidas de limite (`>`): por que manter os "maior que"

Uma medida com relação `>` ("IC50 > 30000 nM") não é um valor pontual: ela dá só
um **limite** — "não conseguimos inibir nem nessa concentração alta" —, ou seja,
um inibidor **fraco de verdade**. Em um problema de **regressão** essas medidas
seriam descartadas por não terem valor exato. Aqui, num problema de
**classificação**, elas são exatamente a classe FRACO — a classe que o viés de
publicação torna escassa, porque poucos artigos relatam moléculas que não
funcionam.

Regra: **manter as relações `>`** e rotulá-las FRACO; **descartar as `<`** (uma
molécula "melhor que" um limite baixo é ambígua para a classe). As relações `=`
seguem normalmente.

In [ ]:
# Célula 12 — Curadoria: descarta medidas de limite '<'
antes = len(curados)
curados = curados[curados["standard_relation"] != "<"].copy()
registrar_etapa("descarta relacao <", antes, len(curados), "relacao '<' (limite inferior, ambiguo)")

# marca quais linhas dao apenas um limite '>' (serao forcadas a FRACO adiante)
eh_apenas_limite = []
for relacao in curados["standard_relation"]:
    eh_apenas_limite.append(relacao in (">", ">>"))
curados["apenas_limite"] = eh_apenas_limite
print("medidas '>' mantidas (viram FRACO nas moleculas que so tem elas):",
      int(curados["apenas_limite"].sum()))

### 2.5 — pIC50 e agregação de duplicatas

O ChEMBL **já fornece o pIC50 pronto**, na coluna `pchembl_value` — é
$-\log_{10}$ da potência em mol/L, curado por eles. Usamos esse valor direto (para
medidas exatas o `pchembl_value` e o cálculo à mão $9 - \log_{10}(\text{nM})$
coincidem, o que serve de verificação).

**E as medidas de limite `>`?** Elas **não recebem pIC50**. Uma medida
"IC50 > 30000 nM" não é um valor pontual — é só um **limite** ("no mínimo tão
fraca quanto isto"). Fabricar um número a partir dela seria inventar precisão que
não existe, e esse número nem é usado: uma molécula que só tem medidas `>` é
rotulada **FRACO por regra** (é um inativo conhecido). O pIC50 só faz falta para
molécula que tem medida **exata** — e essas têm.

Consequência importante: o rótulo de uma molécula vem das suas medidas **exatas**,
quando existem. Uma molécula com um IC50 exato potente **não** vira FRACO só
porque um outro ensaio reportou `>` — as medidas exatas decidem; as de limite só
confirmam a fraqueza de quem não tem nenhuma medida exata.

Agregamos por molécula pela **mediana** das medidas exatas (robusta a outliers) e
**descartamos moléculas cuja dispersão passa de uma unidade log** (exatas que
discordam por mais de 10x não são confiáveis).

In [ ]:
# Célula 13 — Calcula o pIC50 e agrega duplicatas por molécula
# pIC50 apenas das medidas EXATAS: usa o pchembl_value curado; onde faltar
# (raro, medidas exatas sem pchembl) calcula 9 - log10(nM).
pchembl = pd.to_numeric(curados["pchembl_value"], errors="coerce")
pic50_calculado = 9.0 - np.log10(curados["standard_value"])   # = -log10(mol/L)
curados["pic50"] = pchembl.where(pchembl.notna(), pic50_calculado)
# medidas '>' dao apenas um limite: nao entram no pIC50
curados.loc[curados["apenas_limite"], "pic50"] = np.nan

n_exatas = int(curados["pic50"].notna().sum())
print("medidas com pIC50 (exatas):", n_exatas,
      "| de limite (so '>', sem pIC50):", len(curados) - n_exatas)
mascara_ambos = pchembl.notna() & curados["pic50"].notna()
print("diferenca media |pchembl - calculado| (medidas exatas):",
      round((pchembl[mascara_ambos] - pic50_calculado[mascara_ambos]).abs().mean(), 3),
      "(esperado ~0)")

# agrega por molecula usando SO as medidas exatas
grupos = curados.groupby("molecule_chembl_id")
linhas_agregadas = []
for id_molecula, bloco in grupos:
    pic50_exatas = bloco["pic50"].dropna()          # descarta os NaN das medidas de limite
    sem_medida_exata = len(pic50_exatas) == 0       # molecula so tem medidas '>'
    if sem_medida_exata:
        pic50_mediana = np.nan
        dispersao = 0.0                             # sem exatas para discordar
    else:
        pic50_mediana = pic50_exatas.median()
        dispersao = pic50_exatas.max() - pic50_exatas.min()
    smiles = bloco["canonical_smiles"].iloc[0]
    linhas_agregadas.append({
        "molecule_chembl_id": id_molecula,
        "canonical_smiles": smiles,
        "pic50": pic50_mediana,
        "dispersao_log": dispersao,
        "sem_medida_exata": sem_medida_exata,
        "n_medidas": len(bloco),
    })
agregados = pd.DataFrame(linhas_agregadas)
print("moleculas unicas apos agregacao:", len(agregados))
print("  destas, so com medida '>' (serao FRACO por regra):",
      int(agregados["sem_medida_exata"].sum()))

antes = len(agregados)
agregados = agregados[agregados["dispersao_log"] <= 1.0].copy()
registrar_etapa("dispersao <= 1 log", antes, len(agregados),
                "medidas discordam mais de 10x (pIC50 nao confiavel)")

### 2.6 — Tabela de proveniência e funil

Agora a prestação de contas: a tabela cumulativa de tudo que fizemos, e o mesmo
em um gráfico de funil interativo. Nada saiu dos dados sem motivo registrado.

In [ ]:
# Célula 14 — Monta a tabela de proveniência e o funil
tabela_proveniencia = pd.DataFrame(
    proveniencia, columns=["etapa", "n_restante", "n_removido", "motivo"])
print(tabela_proveniencia.to_string(index=False))

figura_funil = go.Figure(go.Funnel(
    y=tabela_proveniencia["etapa"],
    x=tabela_proveniencia["n_restante"],
    textinfo="value+percent initial"))
figura_funil.update_layout(title="Funil de curadoria dos dados", height=420)
figura_funil.show()

## Seção 3 — Os descritores, explicados um a um

Um modelo não "vê" moléculas: vê números. Precisamos transformar cada molécula em
um vetor de números — os **descritores**. Usaremos dois tipos: descritores
físico-químicos interpretáveis e o fingerprint de Morgan (estrutural).

### Descritores físico-químicos

| Descritor | O que mede | Por que importa | Faixa típica |
| --- | --- | --- | --- |
| MW | massa molecular | tamanho; ligado à permeabilidade e à potência bruta | 150–500 |
| LogP | lipofilicidade | equilíbrio água/gordura; permeação de membrana | -1 a 5 |
| TPSA | área polar topológica | ligada à absorção e à passagem pela barreira hematoencefálica | 20–140 |
| HBD | doadores de ligação de H | interações com o alvo; permeabilidade | 0–5 |
| HBA | aceptores de ligação de H | interações com o alvo; solubilidade | 0–10 |
| Ligações rotáveis | flexibilidade | rigidez favorece afinidade e biodisponibilidade | 0–10 |
| Anéis aromáticos | número de anéis aromáticos | empilhamento com resíduos aromáticos do sítio | 0–4 |
| Fração sp3 | carbonos sp3 / carbonos | "tridimensionalidade"; ligada a sucesso clínico | 0–1 |
| Átomos pesados | átomos não-hidrogênio | tamanho bruto; volta a importar na Seção 6 | 10–40 |

A regra dos cinco de Lipinski (MW≤500, LogP≤5, HBD≤5, HBA≤10) resume boa parte
disso para fármacos orais.

In [ ]:
# Célula 15 — Converte SMILES em moléculas e calcula os 9 descritores
# converte cada SMILES em objeto molecula uma unica vez (reaproveitado adiante)
moleculas = []
for smiles in agregados["canonical_smiles"]:
    moleculas.append(Chem.MolFromSmiles(smiles))
agregados["molecula"] = moleculas

# calcula os nove descritores de cada molecula, com um laco explicito
lista_descritores = []
for molecula in agregados["molecula"]:
    descritores = {}
    descritores["MW"] = Descriptors.MolWt(molecula)
    descritores["LogP"] = Descriptors.MolLogP(molecula)
    descritores["TPSA"] = Descriptors.TPSA(molecula)
    descritores["HBD"] = Descriptors.NumHDonors(molecula)
    descritores["HBA"] = Descriptors.NumHAcceptors(molecula)
    descritores["ligacoes_rotaveis"] = Descriptors.NumRotatableBonds(molecula)
    descritores["aneis_aromaticos"] = Descriptors.NumAromaticRings(molecula)
    descritores["fracao_sp3"] = Descriptors.FractionCSP3(molecula)
    descritores["atomos_pesados"] = molecula.GetNumHeavyAtoms()
    lista_descritores.append(descritores)
tabela_descritores = pd.DataFrame(lista_descritores, index=agregados.index)
print("dimensao da tabela de descritores:", tabela_descritores.shape)
tabela_descritores.describe().round(2)

### 3.1 — Correlação entre os descritores (e a questão da escala)

Antes dos fingerprints, vale olhar como os nove descritores se relacionam entre
si. O mapa de calor abaixo mostra a **correlação de Pearson** de cada par: valores
próximos de +1 (ou -1) indicam descritores que carregam quase a mesma informação —
redundância que ajuda a entender o que o modelo de fato tem em mãos. É de se
esperar, por exemplo, que peso molecular, número de átomos pesados e aceptores de
ligação de hidrogênio andem juntos: todos crescem com o tamanho da molécula.

**Normalizar as features não teria sido bom?** Depende do modelo. Métodos baseados
em distância ou gradiente — a regressão logística, a SVM e a rede neural — **são**
sensíveis à escala, e por isso já os embrulhamos num `Pipeline` com
`StandardScaler` (média 0, desvio 1) nas suas próprias células. Já as **árvores** e
a **floresta** decidem por limiares em cada variável isoladamente, então são
**indiferentes** a qualquer reescala monotônica — normalizar não muda nada nelas.
Ou seja: a padronização é aplicada onde importa, dentro de cada pipeline, e omitida
onde é inócua.

In [ ]:
# Célula 16 — Correlação entre os descritores (mapa de calor)
# matriz de correlacao de Pearson entre os 9 descritores
matriz_correlacao = tabela_descritores.corr()

figura_correlacao = px.imshow(
    matriz_correlacao, text_auto=".2f", aspect="auto",
    color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
    title="Correlacao entre os descritores fisico-quimicos")
figura_correlacao.update_layout(height=470, coloraxis_colorbar_title="Pearson")
figura_correlacao.show()

# maior correlacao (em modulo) fora da diagonal, so para nomear a redundancia
correlacao_sem_diagonal = matriz_correlacao.where(~np.eye(len(matriz_correlacao), dtype=bool))
par = correlacao_sem_diagonal.abs().stack().idxmax()
print("par de descritores mais correlacionado:", par,
      "->", round(matriz_correlacao.loc[par[0], par[1]], 2))

### 3.2 — O fingerprint de Morgan, visualmente

Descritores físico-químicos resumem a molécula em poucos números, mas não dizem
*quais subestruturas* ela tem. O **fingerprint de Morgan** faz isso: percorre
cada átomo, olha a vizinhança em círculos de raio crescente, e liga um bit para
cada subestrutura encontrada. O resultado é um vetor binário longo (aqui 2048
bits): 1 se a subestrutura está presente, 0 se não.

Cada bit é, portanto, "esta molécula contém esta pequena subestrutura?". Vamos
ver isso com os olhos: pegamos uma molécula e desenhamos duas subestruturas
(bits) que ela ativa.

In [ ]:
# Célula 17 — Fingerprint de Morgan de uma molécula, ilustrado
RAIO_MORGAN = 2
N_BITS = 2048

# escolhe uma molecula relativamente potente para ilustrar
indice_exemplo = agregados["pic50"].idxmax()
molecula_exemplo = agregados.loc[indice_exemplo, "molecula"]

# calcula o fingerprint guardando quais atomos ativaram cada bit (bitInfo)
info_bits = {}
fp_exemplo = AllChem.GetMorganFingerprintAsBitVect(
    molecula_exemplo, RAIO_MORGAN, nBits=N_BITS, bitInfo=info_bits)
bits_ativos = list(info_bits.keys())
print("molecula de exemplo:", agregados.loc[indice_exemplo, "molecule_chembl_id"])
print("bits ativados nesta molecula:", len(bits_ativos), "de", N_BITS)

# desenha a molecula
Draw.MolToImage(molecula_exemplo, size=(360, 260))

Agora dois ou três bits específicos: cada painel mostra o átomo central
(destacado) e a vizinhança que define aquela subestrutura. É isso que um "1" no
fingerprint significa.

In [ ]:
# Célula 18 — Desenha os bits de Morgan sobre a molécula
bits_para_mostrar = bits_ativos[:3]
tuplas_bits = []
for bit in bits_para_mostrar:
    tuplas_bits.append((molecula_exemplo, bit, info_bits))

imagem_bits = Draw.DrawMorganBits(
    tuplas_bits, molsPerRow=3,
    legends=["bit " + str(bit) for bit in bits_para_mostrar])
imagem_bits

### 3.3 — A tabela onde o treino realmente acontece

Aqui está o ponto que desmistifica "treinar um modelo": juntamos os descritores,
os primeiros bits do fingerprint e o rótulo em **uma tabela numérica**. Treinar é
ajustar uma função que, dessa tabela de entrada, prevê a coluna de rótulo. Não há
mágica — há uma tabela de números.

O rótulo binário: uma molécula que só tem medida `>` é **FRACO** por regra (é um
inativo conhecido, sem pIC50). As demais são **FORTE** se o pIC50 ≥
`LIMIAR_POTENCIA`, senão **FRACO** — decididas pela sua medida exata, como na
Seção 2.5.

In [ ]:
# Célula 19 — Rótulo FORTE/FRACO e matriz de fingerprints
LIMIAR_POTENCIA = 6.0   # pIC50 >= 6 equivale a IC50 <= 1 uM (1000 nM)

# calcula o fingerprint de Morgan de cada molecula e o converte num vetor 0/1
lista_fingerprints = []
for molecula in agregados["molecula"]:
    fp = AllChem.GetMorganFingerprintAsBitVect(molecula, RAIO_MORGAN, nBits=N_BITS)
    vetor = np.zeros((N_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, vetor)
    lista_fingerprints.append(vetor)
matriz_fingerprint = np.array(lista_fingerprints)

# rotulo: 1 = FORTE, 0 = FRACO
rotulo = []
for indice in agregados.index:
    if agregados.loc[indice, "sem_medida_exata"]:
        rotulo.append(0)                       # so tem medida '>': FRACO por regra
    elif agregados.loc[indice, "pic50"] >= LIMIAR_POTENCIA:
        rotulo.append(1)                       # FORTE (decidido pela medida exata)
    else:
        rotulo.append(0)                       # FRACO
agregados["rotulo"] = rotulo

# tabela de treino = descritores + fingerprint + rotulo
tabela_treino = tabela_descritores.copy()
for indice_bit in range(8):
    tabela_treino["fp_" + str(indice_bit)] = matriz_fingerprint[:, indice_bit]
tabela_treino["rotulo"] = agregados["rotulo"].values
print("dimensao da tabela de treino (com fingerprint truncado):", tabela_treino.shape)
tabela_treino.head()

### 3.4 — Dimensões e balanço de classes

Imprimimos o tamanho da matriz completa, a densidade do fingerprint (fração de
bits ligados — fingerprints são esparsos) e quantas moléculas há em cada classe.
O balanço de classes vai definir a linha de base contra a qual julgamos todo
modelo.

In [ ]:
# Célula 20 — Monta X (descritores + fingerprint) e y
# matriz de entrada completa (X) = descritores + fingerprint inteiro
matriz_descritores = tabela_descritores.values.astype(float)
X = np.hstack([matriz_descritores, matriz_fingerprint.astype(float)])
y = agregados["rotulo"].values

nomes_features = list(tabela_descritores.columns)
for indice_bit in range(N_BITS):
    nomes_features.append("fp_" + str(indice_bit))

print("X (moleculas x features):", X.shape)
print("densidade do fingerprint :", round(matriz_fingerprint.mean(), 4),
      "(fracao de bits ligados)")
n_forte = int((y == 1).sum())
n_fraco = int((y == 0).sum())
print("FORTE:", n_forte, "| FRACO:", n_fraco)
maior_classe = max(n_forte, n_fraco)
print("linha de base (chutar a classe maioritaria):",
      round(maior_classe / len(y), 3), "de acuracia")

O mesmo balanço, em um gráfico — para **ver** o tamanho de cada classe. A
barra maior é a linha de base: um modelo que chutasse sempre a classe maior já
acertaria essa fração. Só faz sentido celebrar um modelo que **supere** essa
barra.

In [ ]:
# Célula 21 — Balanço das classes FORTE e FRACO
# tamanho de cada classe, com o numero e a fracao anotados na barra
contagem_classes = [n_fraco, n_forte]
nomes_classes = ["FRACO (0)", "FORTE (1)"]
cores_classes = ["#c0392b", "#3266ad"]
total_moleculas = len(y)

figura_classes, eixo_classes = plt.subplots(figsize=(5, 3.2))
barras = eixo_classes.bar(nomes_classes, contagem_classes, color=cores_classes)
for barra, contagem in zip(barras, contagem_classes):
    fracao = contagem / total_moleculas
    altura = barra.get_height()
    rotulo_barra = str(contagem) + "\n(" + str(round(100 * fracao, 1)) + "%)"
    eixo_classes.text(barra.get_x() + barra.get_width() / 2, altura,
                      rotulo_barra, ha="center", va="bottom", fontsize=9)
eixo_classes.set_ylabel("numero de moleculas")
eixo_classes.set_title("Tamanho de cada classe (n = " + str(total_moleculas) + ")")
eixo_classes.set_ylim(0, max(contagem_classes) * 1.18)
plt.tight_layout()
plt.show()

## Seção 4 — Partição, com representação gráfica

Para estimar honestamente o desempenho, separamos os dados em dois papéis:
**treino** (ajusta o modelo) e **teste** (tocado só no fim, para medir). Não há um
terceiro bloco de "validação": nesta aula não fazemos busca de hiperparâmetros, e a
única rede que precisa parar cedo (early stopping) recorta a própria fração de
validação **de dentro do treino** — não é um bloco à parte.

*Como* separamos importa mais do que o tamanho de cada parte.

A regra, em uma frase: **moléculas de mesmo esqueleto devem ficar do mesmo lado**
(todas no treino ou todas no teste), **nunca separadas** entre treino e teste.

- **Divisão aleatória**: sorteia moléculas uma a uma. O problema é que ela
  **separa** os análogos — séries químicas têm muitos quase-idênticos, e sortear
  faz um cair no treino e o quase-gêmeo no teste. O modelo então "acerta" no teste
  porque já viu um primo no treino, sem ter aprendido nada generalizável — e o
  desempenho **infla**. Essa separação é, na prática, um vazamento.
- **Divisão por esqueleto (Bemis-Murcko)**: agrupa as moléculas pelo esqueleto
  (o núcleo, sem as cadeias laterais) e mantém **cada esqueleto inteiro em um único
  conjunto** — os análogos nunca se separam. Assim o teste contém núcleos que o
  treino **nunca viu**, que é a situação real de uso. É mais dura e mais honesta.

In [ ]:
# Célula 22 — Extrai os esqueletos de Bemis-Murcko
# extrai o esqueleto de Bemis-Murcko (o nucleo, sem cadeias laterais) de cada molecula
esqueletos = []
for molecula in agregados["molecula"]:
    esqueleto = MurckoScaffold.GetScaffoldForMol(molecula)
    esqueletos.append(Chem.MolToSmiles(esqueleto))
agregados["esqueleto"] = esqueletos
print("moleculas:", len(agregados), "| esqueletos distintos:", agregados["esqueleto"].nunique())

### 4.1 — As duas divisões

Implementamos as duas com laços explícitos. Na divisão por esqueleto, ordenamos os
grupos do maior para o menor e vamos preenchendo o treino até atingir a proporção;
o que sobra vira teste — nenhum esqueleto se divide entre conjuntos.

In [ ]:
# Célula 23 — Partição aleatória × por esqueleto (treino/teste)
PROP_TREINO = 0.85   # o resto (0.15) vai para teste
n_total = len(agregados)

# --- divisao POR ESQUELETO: cada esqueleto inteiro fica em um unico conjunto ---
# 1. agrupa os indices das moleculas por esqueleto
grupos_por_esqueleto = {}
for indice in agregados.index:
    chave = agregados.loc[indice, "esqueleto"]
    if chave not in grupos_por_esqueleto:
        grupos_por_esqueleto[chave] = []
    grupos_por_esqueleto[chave].append(indice)

# 2. do maior grupo para o menor, enche o treino ate a proporcao; o resto vira teste
grupos_ordenados = sorted(grupos_por_esqueleto.values(),
                          key=lambda grupo: (-len(grupo), agregados.loc[grupo[0], "esqueleto"]))
idx_treino_esq, idx_teste_esq = [], []
for grupo in grupos_ordenados:
    if len(idx_treino_esq) < PROP_TREINO * n_total:
        idx_treino_esq.extend(grupo)
    else:
        idx_teste_esq.extend(grupo)

# --- divisao ALEATORIA: sorteia moleculas, ignorando o esqueleto ---
indices_embaralhados = list(agregados.index)
np.random.RandomState(SEMENTE).shuffle(indices_embaralhados)
corte_treino = int(PROP_TREINO * n_total)
idx_treino_ale = indices_embaralhados[:corte_treino]
idx_teste_ale = indices_embaralhados[corte_treino:]

print("divisao por esqueleto -> treino", len(idx_treino_esq), "| teste", len(idx_teste_esq))
print("divisao aleatoria     -> treino", len(idx_treino_ale), "| teste", len(idx_teste_ale))

### 4.2 — Diagrama do particionamento

Um esquema desenhado no próprio notebook (Plotly, sem imagem externa): os **dois
blocos** em proporção — treino e teste —, só para ver de relance quanto ficou em
cada lado.

In [ ]:
# Célula 24 — Diagrama do particionamento (treino/teste)
# blocos treino/teste em proporcao
tamanhos = [len(idx_treino_esq), len(idx_teste_esq)]
nomes = ["treino", "teste"]
cores = ["#3266ad", "#c0392b"]
figura_blocos = go.Figure()
for nome, tamanho, cor in zip(nomes, tamanhos, cores):
    figura_blocos.add_trace(go.Bar(
        y=["particao"], x=[tamanho], name=nome, orientation="h", marker_color=cor,
        text=nome + "<br>" + str(tamanho), textposition="inside"))
figura_blocos.update_layout(barmode="stack", height=180,
                            title="Particionamento treino / teste", showlegend=False)
figura_blocos.show()

### 4.3 — O espaço químico das duas divisões, lado a lado

Vamos **ver** as duas divisões. Aqui há uma escolha importante de projeção:
**projetamos o esqueleto (Bemis-Murcko) de cada molécula, não a molécula inteira.**
O motivo é que a divisão é feita *por esqueleto* — então essa é a lente natural
para enxergá-la. Como cada molécula passa a ocupar a posição do seu **núcleo**,
todas as moléculas de um mesmo esqueleto **caem exatamente no mesmo ponto**. É esse
colapso que torna o mecanismo visível.

Cada ponto é pintado pelo **conjunto** a que a molécula pertence (treino ou teste),
uma vez para a divisão aleatória, outra para a por esqueleto. O que procurar:

- na **por esqueleto**, cada aglomerado de pontos coincidentes é de **uma cor só**
  — o esqueleto inteiro foi para um único conjunto; treino e teste nunca partilham
  o mesmo núcleo;
- na **aleatória**, o mesmo aglomerado aparece com **cores misturadas** — os
  análogos de um núcleo foram rasgados entre treino e teste. **Essa mistura é o
  vazamento**, agora visível a olho nu.

**Uma ressalva honesta:** estes dois eixos de PCA ainda resumem só parte da
variação (o valor é impresso abaixo), então não leia distâncias absolutas ao pé da
letra. O que o gráfico mostra bem é o **mecanismo** — a coincidência de pontos por
esqueleto e como cada divisão a colore.

*(E se coloríssemos por classe FORTE/FRACO em vez de por conjunto? As duas classes
apareceriam bem misturadas — dois eixos de PCA não separam as classes, e não
deveriam: se um gráfico 2D já separasse, não precisaríamos de modelo nenhum. A
separação que importa para uma avaliação honesta é entre treino e teste.)*

In [ ]:
# Célula 25 — Projeção PCA do espaço químico das duas divisões
from sklearn.decomposition import PCA

# fingerprint de Morgan do ESQUELETO (nao da molecula inteira) de cada linha.
# moleculas de mesmo esqueleto ficam no mesmo ponto; molecula sem anel tem
# esqueleto vazio -> vetor de zeros (cai na origem).
lista_fp_esqueleto = []
for molecula in agregados["molecula"]:
    mol_esqueleto = MurckoScaffold.GetScaffoldForMol(molecula)
    vetor = np.zeros((N_BITS,), dtype=np.int8)
    if mol_esqueleto is not None and mol_esqueleto.GetNumAtoms() > 0:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol_esqueleto, RAIO_MORGAN, nBits=N_BITS)
        DataStructs.ConvertToNumpyArray(fp, vetor)
    lista_fp_esqueleto.append(vetor)
matriz_fp_esqueleto = np.array(lista_fp_esqueleto)

# projecao 2D dos ESQUELETOS (uma PCA so, compartilhada pelas duas visualizacoes)
pca = PCA(n_components=2, random_state=SEMENTE)
coords = pca.fit_transform(matriz_fp_esqueleto.astype(float))
agregados["pca_x"] = coords[:, 0]
agregados["pca_y"] = coords[:, 1]

# quanto da variacao dos esqueletos estes 2 eixos realmente capturam
var_pc1 = pca.explained_variance_ratio_[0]
var_pc2 = pca.explained_variance_ratio_[1]
var_total = var_pc1 + var_pc2
print("variancia explicada (esqueletos) -> PC1:", round(100 * var_pc1, 1), "%",
      "| PC2:", round(100 * var_pc2, 1), "%",
      "| juntos:", round(100 * var_total, 1), "%")
print("(o resto da estrutura vive nas dezenas de eixos que nao vemos aqui)")

# marca, para cada molecula, a qual conjunto ela pertence em cada divisao
# (uma coluna por divisao, so para colorir a projecao)
for nome_coluna, idx_tr, idx_te in [
    ("conjunto_aleatorio", idx_treino_ale, idx_teste_ale),
    ("conjunto_esqueleto", idx_treino_esq, idx_teste_esq),
]:
    pertence = {}
    for indice in idx_tr: pertence[indice] = "treino"
    for indice in idx_te: pertence[indice] = "teste"
    coluna = []
    for indice in agregados.index:
        coluna.append(pertence.get(indice, "?"))
    agregados[nome_coluna] = coluna

titulo_esq = "Divisao por esqueleto (cada nucleo de uma cor so)"
figura_espaco = make_subplots(rows=1, cols=2,
    subplot_titles=("Divisao aleatoria (nucleos com cores misturadas)", titulo_esq))
mapa_cores = {"treino": "#3266ad", "teste": "#c0392b"}
for coluna_conjunto, col in [("conjunto_aleatorio", 1), ("conjunto_esqueleto", 2)]:
    for nome_conjunto in ["treino", "teste"]:
        sub = agregados[agregados[coluna_conjunto] == nome_conjunto]
        figura_espaco.add_trace(go.Scattergl(
            x=sub["pca_x"], y=sub["pca_y"], mode="markers", name=nome_conjunto,
            marker=dict(size=4, color=mapa_cores[nome_conjunto], opacity=0.5),
            showlegend=(col == 1),
            text=sub["esqueleto"], customdata=sub["pic50"],
            hovertemplate="pIC50=%{customdata:.2f}<br>esqueleto: %{text}<extra></extra>"),
            row=1, col=col)
rotulo_var = "(PC1+PC2 = " + str(round(100 * var_total, 1)) + "% da variancia)"
figura_espaco.update_layout(height=430,
    title="Espaco quimico dos ESQUELETOS (PCA) " + rotulo_var)
figura_espaco.show()

## Seção 5 — Os modelos

Todos os modelos que precisam conversar com o resto do notebook vêm do
**scikit-learn** e compartilham a mesma interface: `fit`, `predict`,
`predict_proba`. Isso não é detalhe: é o que permite avaliar e interpretar
todos eles com o **mesmo** código, sem "adaptadores" que o aluno teria
de decifrar.

A partir daqui usamos a **divisão por esqueleto** como padrão (a honesta).

**A célula abaixo prepara o material dos modelos.** Até aqui a divisão por
esqueleto (Seção 4.1) nos deu duas listas de **rótulos de linha** —
`idx_treino_esq` e `idx_teste_esq` —, que são os índices do DataFrame `agregados`.
Mas `X` e `y` são **arrays NumPy**, indexados por **posição** (0, 1, 2, …), não por
esses rótulos. Então o primeiro passo é **traduzir cada rótulo na sua posição** com
`agregados.index.get_loc(...)`, e só então fatiar `X` e `y` nos dois conjuntos. É
pura logística de indexação — nenhum modelo ainda.

Aproveitamos a mesma célula para duas coisas que todo o resto da seção vai usar:
a **linha de base** do teste (a acurácia de quem chuta sempre a classe
maioritária — o piso que qualquer modelo precisa superar) e dois **dicionários
vazios**, `modelos_treinados` e `tempos_treino`, onde cada modelo vai se guardar
para a comparação única da Seção 5.5.

In [ ]:
# Célula 26 — Recorta X e y de treino e teste
# recorta X e y para treino/teste (divisao por esqueleto).
# X e y estao na ordem de agregados.index; convertemos cada indice na sua posicao.
pos_treino = [agregados.index.get_loc(indice) for indice in idx_treino_esq]
pos_teste  = [agregados.index.get_loc(indice) for indice in idx_teste_esq]
X_treino, y_treino = X[pos_treino], y[pos_treino]
X_teste,  y_teste  = X[pos_teste],  y[pos_teste]
print("treino", X_treino.shape, "| teste", X_teste.shape)

# linha de base do teste: chutar sempre a classe maioritaria do treino
classe_maioritaria = 1 if (y_treino == 1).mean() >= 0.5 else 0
acuracia_base = (y_teste == classe_maioritaria).mean()
print("linha de base (classe maioritaria) no teste:", round(acuracia_base, 3))

# guardaremos aqui cada modelo treinado e seu tempo, para comparar todos juntos em 5.5
modelos_treinados = {}
tempos_treino = {}

### 5.0 — Como vamos avaliar

Treinamos cada modelo em sua própria célula (guardando-o num dicionário
`modelos_treinados`) e, ao final, comparamos **todos de uma vez** com um laço
explícito na Seção 5.5. Assim o código de avaliação aparece **uma vez só**, e você
vê o mesmo cálculo aplicado igualmente a cada modelo.

**Por que MCC, e não RMSE?** Porque isto é uma tarefa de **classificação**: a
resposta é um rótulo (FORTE/FRACO), não um número. O RMSE (raiz do erro quadrático
médio) mede a distância entre valores previstos e observados — faz sentido em
**regressão**, quando o alvo é contínuo, e não teria significado entre duas classes.
Entre as métricas de classificação, evitamos a **acurácia** porque ela se deixa
enganar por classes desbalanceadas (prever sempre a maioria já "acerta" muito). O
**MCC** (coeficiente de correlação de Matthews) leva em conta os quatro quadrantes
da matriz de confusão de uma vez e só fica alto quando o modelo acerta as duas
classes: base = 0 (chute), 1 = perfeito. Ao lado dele reportamos a **AUC**, que
resume o desempenho em todos os limiares de decisão.

### 5.1 — Regressão logística

O modelo mais simples que ainda é um modelo — e a referência contra a qual os
outros são julgados. Ele faz duas coisas: (1) calcula uma **soma ponderada** das
features, $z = w_1 x_1 + w_2 x_2 + \dots + b$ (cada peso $w$ diz o quanto aquela
feature conta, e o sinal, para que lado); (2) passa esse $z$ por uma **sigmoide**,
que espreme qualquer número real para o intervalo $[0, 1]$ — virando a
**probabilidade** de a molécula ser FORTE. Se a probabilidade passa de 0,5, a
predição é FORTE; senão, FRACO. Os pesos são **interpretáveis**: o sinal diz se a
feature empurra para FORTE ou para FRACO, algo que retomaremos na Seção 6.

O gráfico abaixo mostra exatamente essa conversão — a soma ponderada no eixo x, a
probabilidade no eixo y, e o limiar de 0,5 que separa as duas decisões.

In [ ]:
# Célula 27 — Ilustração da sigmoide da regressão logística
# figura didatica: como a regressao logistica transforma a soma ponderada
# (z) numa probabilidade, via sigmoide. NAO usa os dados; ilustra o mecanismo.
z = np.linspace(-6, 6, 300)
probabilidade = 1.0 / (1.0 + np.exp(-z))          # a funcao sigmoide

figura_logistica, eixo_log = plt.subplots(figsize=(6.4, 4.0))
eixo_log.plot(z, probabilidade, color="#3266ad", linewidth=2.5)
eixo_log.axhline(0.5, color="gray", linestyle="--", linewidth=1)
eixo_log.axvline(0.0, color="gray", linestyle=":", linewidth=1)
eixo_log.fill_between(z, 0.5, probabilidade, where=(probabilidade >= 0.5),
                      color="#3266ad", alpha=0.12)
eixo_log.fill_between(z, probabilidade, 0.5, where=(probabilidade < 0.5),
                      color="#c0392b", alpha=0.12)
eixo_log.text(3.1, 0.14, "preve FORTE\n(prob > 0.5)", color="#3266ad",
              ha="center", fontsize=9)
eixo_log.text(-3.1, 0.86, "preve FRACO\n(prob < 0.5)", color="#c0392b",
              ha="center", fontsize=9)
eixo_log.set_xlabel("z = soma ponderada das features  (w . x + b)")
eixo_log.set_ylabel("probabilidade de FORTE = sigmoide(z)")
eixo_log.set_title("Regressao logistica: da soma ponderada a probabilidade")
eixo_log.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.show()

**Como funciona o `Pipeline` do scikit-learn.** A célula de treino não chama
a regressão logística sozinha — ela a embrulha num `Pipeline`, que **encadeia
etapas e as trata como um único modelo**. Aqui são duas etapas nomeadas:
`("escala", StandardScaler())`, que padroniza cada feature (subtrai a média,
divide pelo desvio, deixando tudo na mesma escala — a regressão logística é
sensível a isso), e `("clf", LogisticRegression(...))`, o classificador em si. O
`Pipeline` expõe a mesma interface de qualquer modelo (`fit`, `predict`,
`predict_proba`) e roteia os dados pelos passos na ordem:

- no **`fit`**: o scaler **aprende** a média e o desvio *do treino* e transforma o
  treino; a regressão logística treina sobre esse treino já padronizado;
- no **`predict`**: o scaler **reaplica ao teste os mesmos números que aprendeu no
  treino** (não recalcula) e a regressão logística prediz.

**Por que embrulhar, em vez de padronizar antes?** Para não vazar. Se ajustássemos
o `StandardScaler` no conjunto **todo** antes de dividir, a média e o desvio
carregariam informação do teste para dentro do treino, e a estimativa de
desempenho sairia otimista. Dentro do `Pipeline`, como o `fit` só enxerga
`X_treino`, o scaler nunca "vê" o teste. É o mesmo vazamento da partição (Seção
4.3), agora no nível da transformação das features. Os modelos sensíveis à escala
(5.1 e a SVM da 5.2) usam esse padrão; a floresta (5.3) dispensa o scaler.

In [ ]:
# Célula 28 — Treina a regressão logística
inicio = time.time()
modelo_logistico = Pipeline([
    ("escala", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEMENTE)),
])
modelo_logistico.fit(X_treino, y_treino)
tempo = time.time() - inicio

modelos_treinados["Regressao logistica"] = modelo_logistico
tempos_treino["Regressao logistica"] = tempo
mcc = matthews_corrcoef(y_teste, modelo_logistico.predict(X_teste))
print(f"Regressao logistica treinada em {tempo:.1f}s | MCC no teste = {mcc:.3f}")

### 5.2 — Máquina de vetores de suporte (SVM)

A SVM procura a fronteira que separa as classes com a **maior margem**. Com o
**truque do kernel**, ela traça fronteiras curvas sem calcular explicitamente as
coordenadas em alta dimensão. Vejamos a ideia em 2D, com dados sintéticos não
separáveis por uma reta e a fronteira do kernel RBF traçada sobre eles.

In [ ]:
# Célula 29 — SVM em dados circulares (o truque do kernel, ilustrado)
from sklearn.datasets import make_circles

X_circulos, y_circulos = make_circles(n_samples=300, factor=0.4, noise=0.12,
                                      random_state=SEMENTE)
svm_demo = SVC(kernel="rbf", gamma=2.0, C=1.0, random_state=SEMENTE)
svm_demo.fit(X_circulos, y_circulos)

# malha para desenhar a fronteira de decisao
passo = 0.02
x_min, x_max = X_circulos[:, 0].min() - 0.3, X_circulos[:, 0].max() + 0.3
y_min, y_max = X_circulos[:, 1].min() - 0.3, X_circulos[:, 1].max() + 0.3
grade_x, grade_y = np.meshgrid(np.arange(x_min, x_max, passo),
                               np.arange(y_min, y_max, passo))
pontos_grade = np.c_[grade_x.ravel(), grade_y.ravel()]
decisao = svm_demo.decision_function(pontos_grade).reshape(grade_x.shape)

plt.figure(figsize=(5, 4.5))
plt.contourf(grade_x, grade_y, decisao, levels=20, cmap="RdBu", alpha=0.6)
plt.contour(grade_x, grade_y, decisao, levels=[0], colors="k", linewidths=1.5)
plt.scatter(X_circulos[:, 0], X_circulos[:, 1], c=y_circulos, cmap="RdBu",
            edgecolors="k", s=25)
plt.title("SVM com kernel RBF: fronteira nao linear")
plt.show()

Agora a SVM sobre os dados reais. Como precisamos das probabilidades para a curva
ROC, usamos `probability=True` — o que acrescenta uma calibração interna por
validação cruzada. O kernel RBF sobre milhares de moléculas e 2048 bits é o passo
**mais lento** da aula; treinamos no conjunto de treino **completo**, sem
subamostrar, então esta célula pode levar alguns minutos.

In [ ]:
# Célula 30 — Treina a SVM (kernel RBF, treino completo)
inicio = time.time()
modelo_svm = Pipeline([
    ("escala", StandardScaler()),
    ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=SEMENTE)),
])
modelo_svm.fit(X_treino, y_treino)          # treino completo, sem subamostrar
tempo = time.time() - inicio

modelos_treinados["SVM (RBF)"] = modelo_svm
tempos_treino["SVM (RBF)"] = tempo
mcc = matthews_corrcoef(y_teste, modelo_svm.predict(X_teste))
print(f"SVM (RBF) treinada em {tempo:.1f}s | MCC no teste = {mcc:.3f}")

### 5.3 — Floresta aleatória

Antes da floresta, a **árvore**. Uma árvore de decisão classifica por uma sequência
de perguntas de sim/não sobre uma feature de cada vez ("TPSA > 60?", "tem o bit
512?"). Como ela **escolhe** cada pergunta? Testa todas as features e todos os
cortes possíveis e fica com o que **melhor separa as classes** — o que deixa os
dois lados mais "puros" (mais perto de conter uma classe só, medido pela impureza
de Gini). Feito o corte, repete o processo em cada lado, recursivamente, até os
grupos ficarem puros ou até bater um limite de profundidade. Cada caixa no fim do
caminho é uma **folha**, rotulada pela classe majoritária das moléculas de treino
que ali caíram. Para **prever**, uma molécula nova desce a árvore respondendo às
perguntas e recebe o rótulo da folha onde parar.

Uma árvore fundo demais decora o treino (sobreajuste). Desenhamos uma árvore rasa
(profundidade 3), legível, só para ver o mecanismo. Depois, a **floresta**: muitas
árvores treinadas em reamostras diferentes dos dados e das features, cujo **voto
médio** reduz a variância de uma árvore isolada — o todo generaliza melhor que
qualquer árvore sozinha.

In [ ]:
# Célula 31 — Árvore rasa para visualizar o mecanismo
# arvore rasa, so para visualizar o mecanismo (nao entra na comparacao)
arvore_rasa = DecisionTreeClassifier(max_depth=3, random_state=SEMENTE)
arvore_rasa.fit(X_treino, y_treino)
plt.figure(figsize=(13, 6))
plot_tree(arvore_rasa, max_depth=3, feature_names=nomes_features,
          class_names=["FRACO", "FORTE"], filled=True, fontsize=7, impurity=False)
plt.title("Uma arvore de decisao rasa (profundidade 3)")
plt.show()

In [ ]:
# Célula 32 — Treina a floresta aleatória
inicio = time.time()
modelo_floresta = RandomForestClassifier(
    n_estimators=300, max_depth=None, n_jobs=-1, random_state=SEMENTE)
modelo_floresta.fit(X_treino, y_treino)
tempo = time.time() - inicio

modelos_treinados["Floresta aleatoria"] = modelo_floresta
tempos_treino["Floresta aleatoria"] = tempo
mcc = matthews_corrcoef(y_teste, modelo_floresta.predict(X_teste))
print(f"Floresta aleatoria treinada em {tempo:.1f}s | MCC no teste = {mcc:.3f}")

Uma pergunta natural sobre a floresta: **quantas árvores bastam?** Crescemos a
floresta em etapas (com `warm_start`, que só acrescenta as árvores novas em vez de
recomeçar) e medimos o MCC no teste a cada etapa. A curva **estabiliza** cedo,
num platô: já com poucas dezenas de árvores o MCC praticamente não muda, e
acrescentar mais só custa tempo. É o diagnóstico que justifica as 300 árvores
usadas acima — uma folga confortável dentro do platô.

In [ ]:
# Célula 33 — MCC da floresta conforme se adicionam árvores
# cresce a floresta em etapas e mede o MCC no teste a cada etapa
floresta_incremental = RandomForestClassifier(
    n_estimators=10, warm_start=True, n_jobs=-1, random_state=SEMENTE)

n_arvores_etapas = list(range(20, 301, 20))
curva_floresta = []
for n_arvores in n_arvores_etapas:
    floresta_incremental.set_params(n_estimators=n_arvores)
    floresta_incremental.fit(X_treino, y_treino)     # warm_start: so cresce as novas arvores
    curva_floresta.append(matthews_corrcoef(y_teste, floresta_incremental.predict(X_teste)))
print("MCC com 20 arvores:", round(curva_floresta[0], 3),
      "| com 300 arvores:", round(curva_floresta[-1], 3))

In [ ]:
# Célula 34 — Curva MCC × nº de árvores (Plotly)
figura_floresta = go.Figure(go.Scatter(
    x=n_arvores_etapas, y=curva_floresta, mode="lines+markers", line=dict(color="#1a7a4a")))
figura_floresta.update_layout(
    title="Floresta aleatoria: MCC no teste conforme se adicionam arvores",
    xaxis_title="numero de arvores", yaxis_title="MCC", height=360)
figura_floresta.show()
print("MCC estabiliza em torno de", round(curva_floresta[-1], 3),
      "- mais arvores alem disso quase nao mudam nada")

### 5.4a — Rede neural (scikit-learn)

Uma rede neural encadeia **camadas**: cada camada combina as entradas com pesos e
aplica uma **ativação** não linear. Termos, na primeira vez que aparecem:
**época** é uma passada por todos os dados; **minilote** é um punhado de exemplos
processado por vez; **taxa de aprendizado** é o tamanho do passo com que os pesos
são ajustados. O `MLPClassifier` faz tudo isso por dentro.

Usamos `early_stopping=True`: o treino para quando o desempenho numa fração de
validação deixa de melhorar. Plotamos as duas curvas por época — perda de treino
(`loss_curve_`) e desempenho na validação (`validation_scores_`). O descolamento
entre elas é o **sobreajuste** ficando visível.

In [ ]:
# Célula 35 — Treina a rede neural (MLP do scikit-learn)
inicio = time.time()
modelo_rede = MLPClassifier(
    hidden_layer_sizes=(128, 64), activation="relu",
    early_stopping=True, n_iter_no_change=10, max_iter=200,
    random_state=SEMENTE)
modelo_rede.fit(X_treino, y_treino)
tempo = time.time() - inicio

modelos_treinados["Rede neural (MLP)"] = modelo_rede
tempos_treino["Rede neural (MLP)"] = tempo
mcc = matthews_corrcoef(y_teste, modelo_rede.predict(X_teste))
print(f"Rede neural (MLP) treinada em {tempo:.1f}s | MCC no teste = {mcc:.3f}")

figura_rede = make_subplots(specs=[[{"secondary_y": True}]])
epocas = list(range(1, len(modelo_rede.loss_curve_) + 1))
figura_rede.add_trace(go.Scatter(x=epocas, y=modelo_rede.loss_curve_,
                                 name="perda de treino", line=dict(color="#3266ad")),
                      secondary_y=False)
figura_rede.add_trace(go.Scatter(x=list(range(1, len(modelo_rede.validation_scores_) + 1)),
                                 y=modelo_rede.validation_scores_,
                                 name="acuracia na validacao", line=dict(color="#7e9603")),
                      secondary_y=True)
figura_rede.update_layout(title="MLP: perda de treino x desempenho na validacao", height=380)
figura_rede.show()

### 5.4b — A mesma rede, aberta, em PyTorch (não opcional)

A parte (a) chama `fit` e o treino acontece numa caixa preta. Aqui abrimos a
caixa: reimplementamos **a mesma** rede em PyTorch e escrevemos o laço de treino
à mão. O objetivo não é um modelo melhor — é **ver o que o `fit` faz por dentro**.

Antes do código, os conceitos. Um **gradiente** é a direção em que a perda mais
cresce; andamos no sentido oposto para diminuí-la. A cada minilote, quatro passos:
zerar os gradientes acumulados, calcular a saída (passagem direta), medir a perda,
e então `loss.backward()` — que calcula os gradientes por retropropagação —
seguido de `optimizer.step()`, que dá o passo. `backward()` vem **depois** da
perda (precisa dela para saber o erro) e **antes** do passo (que usa os gradientes
que ela produziu).

Esta rede não conversa com o resto do notebook, então não precisa de invólucro:
fatiamos os tensores em minilotes com um laço sobre índices embaralhados, sem
`DataLoader`.

In [ ]:
# Célula 36 — Prepara tensores e monta a mesma rede em PyTorch
# preparo: padroniza (mesma ideia dos pipelines) e converte para tensores
from sklearn.preprocessing import StandardScaler as _Escala
escala_torch = _Escala().fit(X_treino)
Xt_treino = torch.tensor(escala_torch.transform(X_treino), dtype=torch.float32)
yt_treino = torch.tensor(y_treino, dtype=torch.long)
Xt_teste = torch.tensor(escala_torch.transform(X_teste), dtype=torch.float32)

class RedeMLP(nn.Module):
    """Mesma arquitetura da parte (a): 2 camadas ocultas ReLU, saida com 2 classes."""
    def __init__(self, n_entradas):
        super().__init__()
        self.camada1 = nn.Linear(n_entradas, 128)
        self.camada2 = nn.Linear(128, 64)
        self.saida = nn.Linear(64, 2)
        self.ativacao = nn.ReLU()

    def forward(self, entrada):
        oculta1 = self.ativacao(self.camada1(entrada))
        oculta2 = self.ativacao(self.camada2(oculta1))
        return self.saida(oculta2)

torch.manual_seed(SEMENTE)
rede_torch = RedeMLP(X_treino.shape[1]).to(DISPOSITIVO)
funcao_perda = nn.CrossEntropyLoss()
otimizador = torch.optim.Adam(rede_torch.parameters(), lr=0.001)
print("rede criada em", DISPOSITIVO, "| parametros:",
      sum(p.numel() for p in rede_torch.parameters()))

**Vamos desenhar a arquitetura que acabamos de montar** — o diagrama real,
gerado a partir do próprio modelo, não um desenho decorativo. Para modelos PyTorch,
o pacote mais prático hoje é o **`torchview`** (`draw_graph`): ele passa um tensor
de exemplo pela rede, rastreia as operações e desenha as camadas **com as formas
dos tensores** em cada ponto — do vetor de entrada até os 2 logits de saída.
(Alternativas, se quiser explorar: o **`torchinfo`** dá uma tabela textual no
estilo do `model.summary()` do Keras; o **`torchviz`** desenha o grafo de
*autograd*, isto é, as operações da retropropagação, útil para depurar em vez da
arquitetura.)

O `torchview` não vem no Colab, então a célula abaixo o **instala** (rápido) e
gera o diagrama. Antes dele, imprimimos um resumo camada a camada com a contagem
de parâmetros.

In [ ]:
# Célula 37 — Instala o torchview e desenha a arquitetura da rede
# instala o torchview se faltar (mesmo padrao robusto da Secao 0)
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("torchview") is None:
    print("instalando torchview...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchview"], check=True)

# resumo textual camada a camada, direto do modelo
print("camada a camada:")
for nome_camada, modulo in rede_torch.named_children():
    n_parametros = sum(p.numel() for p in modulo.parameters())
    print("  ", nome_camada, "->", modulo, "| parametros:", n_parametros)
print("total de parametros:", sum(p.numel() for p in rede_torch.parameters()))

# diagrama real da arquitetura, com as formas dos tensores (renderiza no Colab)
from torchview import draw_graph
grafo_rede = draw_graph(rede_torch, input_size=(1, X_treino.shape[1]),
                        device=str(DISPOSITIVO), graph_name="RedeMLP")
grafo_rede.visual_graph   # ultima expressao: o Colab desenha o diagrama aqui

O laço de treino, comentado passo a passo, com uma barra `tqdm` para acompanhar as
épocas. Guardamos a perda média de cada época numa lista para desenhá-la em Plotly
logo depois, ao lado da `loss_curve_` do `MLPClassifier` da parte (a) — assim as
duas implementações da mesma rede aparecem no mesmo gráfico.

In [ ]:
# Célula 38 — Treina a rede em PyTorch (laço explícito)
N_EPOCAS = 40
TAM_MINILOTE = 64

perda_por_epoca_torch = []
n_treino = Xt_treino.shape[0]
inicio_torch = time.time()
for epoca in tqdm(range(N_EPOCAS), desc="treino PyTorch"):
    rede_torch.train()
    # embaralha os indices a cada epoca e fatia em minilotes
    ordem = torch.randperm(n_treino)
    perda_acumulada = 0.0
    n_lotes = 0
    for inicio_lote in range(0, n_treino, TAM_MINILOTE):
        indices_lote = ordem[inicio_lote:inicio_lote + TAM_MINILOTE]
        entradas = Xt_treino[indices_lote].to(DISPOSITIVO)
        alvos = yt_treino[indices_lote].to(DISPOSITIVO)

        otimizador.zero_grad()                 # 1. zera gradientes acumulados
        saidas = rede_torch(entradas)          # 2. passagem direta
        perda = funcao_perda(saidas, alvos)    # 3. mede a perda
        perda.backward()                       # 4. retropropaga (calcula gradientes)
        otimizador.step()                      # 5. da o passo (ajusta os pesos)

        perda_acumulada = perda_acumulada + perda.item()
        n_lotes = n_lotes + 1
    perda_por_epoca_torch.append(perda_acumulada / n_lotes)
tempo_torch = time.time() - inicio_torch
print("treino PyTorch concluido em", round(tempo_torch, 1), "s |",
      "perda final:", round(perda_por_epoca_torch[-1], 4))

Registro permanente em Plotly: as duas curvas de perda sobrepostas.
Elas **não coincidem**, embora sejam a mesma arquitetura — inicialização de pesos
diferente, otimizadores diferentes, ordem dos minilotes diferente. Que dois
códigos corretos para o mesmo modelo deem curvas diferentes é, em si, uma lição
sobre estocasticidade em aprendizado de máquina.

In [ ]:
# Célula 39 — Curva de perda da rede (PyTorch × MLP, em Plotly)
figura_perdas = go.Figure()
figura_perdas.add_trace(go.Scatter(
    x=list(range(1, len(perda_por_epoca_torch) + 1)), y=perda_por_epoca_torch,
    name="PyTorch (laco a mao)", line=dict(color="#c0392b")))
figura_perdas.add_trace(go.Scatter(
    x=list(range(1, len(modelo_rede.loss_curve_) + 1)), y=modelo_rede.loss_curve_,
    name="MLPClassifier (sklearn)", line=dict(color="#3266ad")))
figura_perdas.update_layout(title="Mesma rede, duas implementacoes: perda por epoca",
                            xaxis_title="epoca", yaxis_title="perda", height=380)
figura_perdas.show()

### 5.5 — Comparação de todos os modelos

Agora a **avaliação em um laço só**: percorremos os modelos guardados e aplicamos a
cada um exatamente o mesmo cálculo de métricas. Como os modelos do scikit-learn
compartilham a interface (`predict`, `predict_proba`), o mesmo código serve para
todos. A rede em PyTorch (5.4b) não é um estimador do scikit-learn, então a
acrescentamos **à mão** ao final, para que ela também apareça na tabela e nos
gráficos. Guardamos tudo em `resultados`.

In [ ]:
# Célula 40 — Avalia todos os modelos com o mesmo código
# avalia todos os modelos do scikit-learn com o MESMO codigo, num laco explicito
resultados = {}
for nome, modelo in modelos_treinados.items():
    predito = modelo.predict(X_teste)
    if hasattr(modelo, "predict_proba"):
        proba = modelo.predict_proba(X_teste)[:, 1]
    else:
        proba = modelo.decision_function(X_teste)
    relatorio = classification_report(y_teste, predito, output_dict=True,
                                      target_names=["FRACO", "FORTE"], zero_division=0)
    resultados[nome] = {
        "MCC": matthews_corrcoef(y_teste, predito),
        "AUC": roc_auc_score(y_teste, proba),
        "prec_FORTE": relatorio["FORTE"]["precision"],
        "rec_FORTE": relatorio["FORTE"]["recall"],
        "prec_FRACO": relatorio["FRACO"]["precision"],
        "rec_FRACO": relatorio["FRACO"]["recall"],
        "tempo_s": tempos_treino[nome],
        "proba": proba, "predito": predito,
    }

# a rede em PyTorch (5.4b) nao e um estimador sklearn: entra a mao, com o mesmo calculo
rede_torch.eval()
with torch.no_grad():
    proba_torch = torch.softmax(rede_torch(Xt_teste.to(DISPOSITIVO)), dim=1)[:, 1].cpu().numpy()
predito_torch = (proba_torch >= 0.5).astype(int)
relatorio_torch = classification_report(y_teste, predito_torch, output_dict=True,
                                        target_names=["FRACO", "FORTE"], zero_division=0)
resultados["Rede neural (PyTorch)"] = {
    "MCC": matthews_corrcoef(y_teste, predito_torch),
    "AUC": roc_auc_score(y_teste, proba_torch),
    "prec_FORTE": relatorio_torch["FORTE"]["precision"],
    "rec_FORTE": relatorio_torch["FORTE"]["recall"],
    "prec_FRACO": relatorio_torch["FRACO"]["precision"],
    "rec_FRACO": relatorio_torch["FRACO"]["recall"],
    "tempo_s": tempo_torch,
    "proba": proba_torch, "predito": predito_torch,
}

for nome in resultados:
    print(f"{nome:24s}: MCC={resultados[nome]['MCC']:.3f}  AUC={resultados[nome]['AUC']:.3f}"
          f"  (base MCC=0.000, AUC=0.500)")

Uma tabela e um gráfico com MCC, AUC, precisão e revocação por classe e
tempo de treino; depois as curvas ROC e de precisão-revocação sobrepostas e as
matrizes de confusão lado a lado. Nenhum número aparece sozinho: a linha de base
(MCC 0, AUC 0,5) está sempre à vista.

In [ ]:
# Célula 41 — Tabela comparativa das métricas
tabela_comparacao = pd.DataFrame(resultados).T[
    ["MCC", "AUC", "prec_FORTE", "rec_FORTE", "prec_FRACO", "rec_FRACO", "tempo_s"]]
tabela_comparacao = tabela_comparacao.astype(float).round(3)
print(tabela_comparacao.to_string())

figura_barras = go.Figure()
for metrica, cor in [("MCC", "#3266ad"), ("AUC", "#7e9603")]:
    figura_barras.add_trace(go.Bar(x=tabela_comparacao.index,
                                   y=tabela_comparacao[metrica], name=metrica,
                                   marker_color=cor))
figura_barras.add_hline(y=0.5, line_dash="dash", line_color="#c0392b",
                        annotation_text="AUC base = 0,5")
figura_barras.update_layout(barmode="group", title="Comparacao de modelos (MCC e AUC)",
                            height=380)
figura_barras.show()

In [ ]:
# Célula 42 — Curvas ROC e precisão-revocação sobrepostas
# curvas ROC e precisao-revocacao sobrepostas
figura_curvas = make_subplots(rows=1, cols=2, subplot_titles=("ROC", "Precisao-Revocacao"))
for nome in resultados:
    proba = resultados[nome]["proba"]
    fpr, tpr, _ = roc_curve(y_teste, proba)
    figura_curvas.add_trace(go.Scatter(x=fpr, y=tpr, name=nome, mode="lines"), row=1, col=1)
    precisao, revocacao, _ = precision_recall_curve(y_teste, proba)
    figura_curvas.add_trace(go.Scatter(x=revocacao, y=precisao, name=nome,
                                       mode="lines", showlegend=False), row=1, col=2)
figura_curvas.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                                   line=dict(dash="dash", color="gray"),
                                   name="acaso"), row=1, col=1)
figura_curvas.update_xaxes(title_text="falsos positivos (1 - especificidade)", row=1, col=1)
figura_curvas.update_yaxes(title_text="verdadeiros positivos (revocacao)", row=1, col=1)
figura_curvas.update_xaxes(title_text="revocacao", row=1, col=2)
figura_curvas.update_yaxes(title_text="precisao", row=1, col=2)
figura_curvas.update_layout(height=430, title="ROC e Precisao-Revocacao (teste)")
figura_curvas.show()

In [ ]:
# Célula 43 — Matrizes de confusão lado a lado
# matrizes de confusao lado a lado
nomes_modelos = list(resultados.keys())
figura_conf, eixos = plt.subplots(1, len(nomes_modelos),
                                  figsize=(3.2 * len(nomes_modelos), 3))
for eixo, nome in zip(eixos, nomes_modelos):
    matriz = confusion_matrix(y_teste, resultados[nome]["predito"])
    eixo.imshow(matriz, cmap="Greens")
    eixo.set_title(nome, fontsize=9)
    eixo.set_xticks([0, 1]); eixo.set_xticklabels(["FRACO", "FORTE"], fontsize=8)
    eixo.set_yticks([0, 1]); eixo.set_yticklabels(["FRACO", "FORTE"], fontsize=8)
    for i in range(2):
        for j in range(2):
            eixo.text(j, i, matriz[i, j], ha="center", va="center", fontsize=11)
plt.tight_layout(); plt.show()

## Seção 6 — Interpretabilidade e o confundimento por tamanho

Um modelo pode acertar pelo motivo errado. Aqui investigamos uma hipótese
concreta: e se os modelos estiverem aprendendo majoritariamente **tamanho
molecular** em vez de química específica de reconhecimento? Potência bruta
correlaciona-se com número de átomos pesados — moléculas maiores fazem mais
contatos — e é por isso que a área usa eficiência de ligante em vez de potência
bruta. **Não afirmamos o resultado de antemão**: testamos.

In [ ]:
# Célula 44 — Correlação tamanho × potência (confundimento)
from scipy.stats import spearmanr

atomos_pesados = tabela_descritores["atomos_pesados"].values
correlacao, _ = spearmanr(atomos_pesados, agregados["pic50"].values)
figura_tamanho = px.scatter(
    x=atomos_pesados, y=agregados["pic50"].values,
    labels={"x": "atomos pesados", "y": "pIC50"},
    title="pIC50 x tamanho molecular (Spearman = " + str(round(correlacao, 3)) + ")",
    opacity=0.4)
figura_tamanho.update_traces(marker=dict(size=4, color="#3266ad"))
figura_tamanho.show()
print("correlacao de Spearman (tamanho x potencia):", round(correlacao, 3))

### 6.1 — Modelo-controle de uma única feature

O teste decisivo: um classificador que usa **apenas** o número de átomos pesados.
Se o MCC dele for próximo ao dos modelos completos, então o fingerprint —
toda a química fina — está contribuindo pouco, e a conclusão da aula muda.

In [ ]:
# Célula 45 — Modelo-controle de uma única feature (tamanho)
X_treino_tamanho = tabela_descritores.loc[idx_treino_esq, ["atomos_pesados"]].values
X_teste_tamanho = tabela_descritores.loc[idx_teste_esq, ["atomos_pesados"]].values

modelo_tamanho = LogisticRegression(max_iter=1000, random_state=SEMENTE)
modelo_tamanho.fit(X_treino_tamanho, y_treino)
predito_tamanho = modelo_tamanho.predict(X_teste_tamanho)
mcc_tamanho = matthews_corrcoef(y_teste, predito_tamanho)
mcc_completo = resultados["Floresta aleatoria"]["MCC"]
print("MCC so com tamanho     :", round(mcc_tamanho, 3))
print("MCC floresta completa  :", round(mcc_completo, 3))
print("diferenca              :", round(mcc_completo - mcc_tamanho, 3))

### 6.2 — Importância por permutação

Qual feature o modelo realmente usa? A **importância por permutação** embaralha
uma feature de cada vez e mede quanto o desempenho cai: se cai muito, a feature
importa. Ela é preferível à importância por impureza da floresta, que é
**enviesada** a favor de variáveis com muitos valores distintos (como os
descritores contínuos) contra as binárias (os bits).

Permutar os 2048 bits **um a um** é caro e enganoso: cada bit isolado quase não
muda o resultado, então todos apareceriam como "sem importância" — e daria a
falsa impressão de que o fingerprint não conta. A saída correta é permutar o
fingerprint **como um bloco**: embaralhamos as linhas de todas as 2048 colunas de
uma vez (mantendo cada fingerprint interno intacto, mas quebrando sua associação
com o rótulo) e medimos a queda. Assim comparamos, na mesma escala, a importância
de **cada descritor** e a do **fingerprint inteiro**. Fazemos tudo com laços
explícitos.

In [ ]:
# Célula 46 — Importância por permutação (descritores × fingerprint)
gerador_perm = np.random.RandomState(SEMENTE)
n_amostra_perm = min(400, len(X_teste))
amostra_perm = gerador_perm.choice(len(X_teste), n_amostra_perm, replace=False)
X_perm = X_teste[amostra_perm].copy()
y_perm = y_teste[amostra_perm]

# MCC de referencia (sem permutar nada)
mcc_referencia = matthews_corrcoef(y_perm, modelo_floresta.predict(X_perm))

# permuta cada coluna de descritor 3 vezes e mede a queda media de MCC
n_descritores = len(tabela_descritores.columns)
importancia_descritores = []
for indice_coluna in range(n_descritores):
    quedas = []
    for repeticao in range(3):
        X_embaralhado = X_perm.copy()
        coluna = X_embaralhado[:, indice_coluna].copy()
        gerador_perm.shuffle(coluna)
        X_embaralhado[:, indice_coluna] = coluna
        mcc_permutado = matthews_corrcoef(y_perm, modelo_floresta.predict(X_embaralhado))
        quedas.append(mcc_referencia - mcc_permutado)
    importancia_descritores.append(float(np.mean(quedas)))

# permuta o FINGERPRINT como um bloco: embaralha as linhas das colunas de bits
# juntas (cada fingerprint fica intacto, mas some a associacao com o rotulo)
colunas_fp = list(range(n_descritores, X_perm.shape[1]))
quedas_fp = []
for repeticao in range(3):
    X_embaralhado = X_perm.copy()
    ordem_linhas = gerador_perm.permutation(len(X_perm))
    X_embaralhado[:, colunas_fp] = X_perm[np.ix_(ordem_linhas, colunas_fp)]
    mcc_permutado = matthews_corrcoef(y_perm, modelo_floresta.predict(X_embaralhado))
    quedas_fp.append(mcc_referencia - mcc_permutado)
importancia_fp = float(np.mean(quedas_fp))

# junta os 9 descritores + o bloco do fingerprint na mesma escala
nomes = list(tabela_descritores.columns) + ["fingerprint (2048 bits)"]
valores = importancia_descritores + [importancia_fp]
cores = ["#7e9603"] * n_descritores + ["#3266ad"]     # fingerprint em azul
ordem_importancia = np.argsort(valores)
figura_importancia = go.Figure(go.Bar(
    x=[valores[i] for i in ordem_importancia],
    y=[nomes[i] for i in ordem_importancia],
    orientation="h", marker_color=[cores[i] for i in ordem_importancia]))
figura_importancia.update_layout(
    title="Importancia por permutacao: descritores x fingerprint (bloco)",
    height=400, xaxis_title="queda media de MCC ao embaralhar")
figura_importancia.show()
print("queda de MCC ao embaralhar o fingerprint inteiro:", round(importancia_fp, 3))
print("maior queda entre os descritores:", round(max(importancia_descritores), 3))

### 6.3 — SHAP sobre a floresta

O SHAP atribui a cada feature, para cada molécula, o quanto ela empurrou a predição
para FORTE ou FRACO — com uma base teórica (valores de Shapley) que reparte a
predição de forma justa entre as features.

Aqui um cuidado com a **representação** faz toda a diferença. Os gráficos de SHAP
mostram uma linha por feature; com os **2048 bits do fingerprint** isso vira um
paredão ilegível de barras `fp_1358`, `fp_746`… — e, pior, no *beeswarm* a cor é o
**valor** da feature, que nos bits binários e esparsos é quase sempre 0 (tudo azul).
A saída é **colapsar o fingerprint num único eixo**: como o SHAP é **aditivo**,
somamos as contribuições dos 2048 bits e obtemos a contribuição do **bloco inteiro**
por molécula. Ficamos com **10 features legíveis** — os 9 descritores mais o
fingerprint agregado — e as mostramos em duas vistas, **uma por célula**:

1. um **gráfico de barras** com a importância média (|SHAP| médio) de cada uma;
2. um **beeswarm** das mesmas 10: nos descritores a cor (valor baixo → alto) conta a
   direção do efeito (por exemplo, se moléculas maiores tendem a FORTE); no eixo do
   fingerprint o "valor" é o número de bits ligados, e ele **costuma dominar** o
   espalhamento — ecoando a importância por permutação (6.2).

Depois abrimos duas moléculas individualmente com o **gráfico de cascata**
(*waterfall*). Restringimos a uma amostra do teste porque o SHAP é custoso.

In [ ]:
# Célula 47 — SHAP: importância das features agrupadas (barra)
import shap

n_amostra_shap = min(120, len(X_teste))
amostra_shap = gerador_perm.choice(len(X_teste), n_amostra_shap, replace=False)
# DataFrame com nomes das features: o SHAP os usa nos rotulos dos graficos
X_shap = pd.DataFrame(X_teste[amostra_shap], columns=nomes_features)

# IMPORTANTE: usamos feature_perturbation="interventional" + model_output="probability",
# com um "fundo" (amostra do treino). E a combinacao que garante valores SHAP na
# escala de PROBABILIDADE (0 a 1) e passa a checagem de aditividade (deixada LIGADA).
# O modo padrao (tree_path_dependent) esta quebrado em versoes recentes do
# scikit-learn e produziria valores fora de escala (milhares em vez de ~0-1).
fundo_shap = X_treino[gerador_perm.choice(len(X_treino),
                                          min(100, len(X_treino)), replace=False)]
explicador = shap.TreeExplainer(modelo_floresta, data=fundo_shap,
                                feature_perturbation="interventional",
                                model_output="probability")
explicacao = explicador(X_shap)                 # checagem de aditividade ligada (default)
explicacao_forte = explicacao[:, :, 1]          # contribuicoes para a classe FORTE

# colapsa os 2048 bits do fingerprint num UNICO eixo: soma as contribuicoes SHAP
# (aditivas) e usa o numero de bits ligados como "valor" para a cor. Sobram 10
# features legiveis (9 descritores + o fingerprint agregado).
valores_agrupados = np.hstack([
    explicacao_forte.values[:, 0:9],
    explicacao_forte.values[:, 9:].sum(axis=1, keepdims=True)])
dados_agrupados = np.hstack([
    explicacao_forte.data[:, 0:9],
    explicacao_forte.data[:, 9:].sum(axis=1, keepdims=True)])   # nº de bits ligados
explicacao_agrupada = shap.Explanation(
    values=valores_agrupados, base_values=explicacao_forte.base_values,
    data=dados_agrupados,
    feature_names=nomes_features[:9] + ["fingerprint (Σ 2048 bits)"])

# vista 1 -- barra: importancia media |SHAP| das 10 features agrupadas
shap.plots.bar(explicacao_agrupada, max_display=10, show=True)

E a mesma informação num **beeswarm** das 10 features (uma célula só para ela): cada
ponto é uma molécula; a posição é a contribuição SHAP e a cor é o valor da feature —
informativa nos descritores, e igual ao número de bits ligados no eixo do fingerprint.

In [ ]:
# Célula 48 — SHAP: beeswarm das features agrupadas
# vista 2 -- beeswarm das mesmas 10 features agrupadas
shap.plots.beeswarm(explicacao_agrupada, max_display=10, show=True)

Agora duas moléculas individuais, cada uma com seu **gráfico de cascata** — nas
**mesmas 10 features agrupadas**: as barras partem do valor-base (a predição média)
e somam, uma feature de cada vez, até a probabilidade daquela molécula. Vermelho
empurra para FORTE, azul para FRACO. O fingerprint entra como **um bloco só (Σ)**,
então não aparecem bits `fp_XXX` individuais — é a leitura molécula a molécula do
mesmo sinal que os resumos acima apontam.

In [ ]:
# Célula 49 — Explicações SHAP individuais (waterfall)
# explica duas moleculas individuais com o waterfall, usando as MESMAS 10 features
# agrupadas (fingerprint colapsado num bloco) -> sem bits fp_XXX soltos.
from IPython.display import display

for posicao_no_shap in [0, 1]:
    indice_molecula = idx_teste_esq[amostra_shap[posicao_no_shap]]
    print("molecula:", agregados.loc[indice_molecula, "molecule_chembl_id"])
    display(Draw.MolToImage(agregados.loc[indice_molecula, "molecula"], size=(320, 240)))
    shap.plots.waterfall(explicacao_agrupada[posicao_no_shap], max_display=10, show=True)

## Seção 7 — Domínio de aplicabilidade

Um modelo só deveria opinar sobre moléculas parecidas com as que viu. Nesta seção
construímos essa noção de forma quantitativa: medimos, para cada molécula de teste,
quão próxima ela está do treino, definimos um **limiar** derivado dos próprios dados
e vemos o compromisso entre **cobertura** (opinar sobre mais moléculas) e **acerto**
(estar certo onde opina). É essa régua que, na Seção 8, permite ao classificador
**abster-se** quando não tem base.

### 7.1 — Medindo a proximidade ao treino

Medimos, para cada molécula de teste, a **similaridade de Tanimoto máxima** contra o
treino.

Antes, um esclarecimento que costuma confundir: **Morgan e Tanimoto não são duas
representações concorrentes.** O fingerprint de Morgan (Seção 3) é a
**representação** — o vetor de bits que descreve a molécula, e que serve de
*feature* para os modelos. A similaridade de Tanimoto é uma **medida** de quão
parecidos são **dois desses fingerprints** (interseção sobre união dos bits
ligados). Ou seja, calculamos Tanimoto **sobre** os mesmos fingerprints de Morgan;
um é a representação, o outro é a régua que compara duas representações. Aqui a régua
serve para o domínio de aplicabilidade, não para treinar.

O limiar que separa "dentro" de "fora" do domínio é derivado dos próprios dados — um
**percentil baixo** das similaridades internas do treino — e não arbitrado. Usamos o
**percentil 2**: uma molécula menos conectada ao espaço químico do que 98% do treino
é considerada fora do domínio. É uma escolha deliberadamente **um pouco permissiva**
(o comum seria o percentil 5, mais estrito), para o modelo opinar sobre mais
moléculas — ao custo de arriscar palpite em casos um pouco mais atípicos. É um botão
explícito: `PERCENTIL_DOMINIO`.

In [ ]:
# Célula 50 — Fingerprints do RDKit para o domínio de aplicabilidade
# fingerprints de Morgan (objetos do RDKit) do treino e do teste, para o Tanimoto
fp_treino_rdkit = []
for indice in idx_treino_esq:
    molecula = agregados.loc[indice, "molecula"]
    fp_treino_rdkit.append(AllChem.GetMorganFingerprintAsBitVect(molecula, RAIO_MORGAN, nBits=N_BITS))
fp_teste_rdkit = []
for indice in idx_teste_esq:
    molecula = agregados.loc[indice, "molecula"]
    fp_teste_rdkit.append(AllChem.GetMorganFingerprintAsBitVect(molecula, RAIO_MORGAN, nBits=N_BITS))

# para cada molecula de teste, a MAIOR similaridade de Tanimoto contra o treino
similaridade_teste = []
for fp in fp_teste_rdkit:
    similaridades = DataStructs.BulkTanimotoSimilarity(fp, fp_treino_rdkit)
    similaridade_teste.append(max(similaridades))
similaridade_teste = np.array(similaridade_teste)

# limiar = percentil baixo das similaridades treino-treino (auto-similaridade excluida)
similaridade_treino_interna = []
for i in range(len(fp_treino_rdkit)):
    sims = DataStructs.BulkTanimotoSimilarity(fp_treino_rdkit[i], fp_treino_rdkit)
    sims[i] = -1.0                 # ignora a similaridade da molecula consigo mesma
    similaridade_treino_interna.append(max(sims))
PERCENTIL_DOMINIO = 2              # menor = dominio mais amplo (mais moleculas 'dentro')
LIMIAR_DOMINIO = float(np.percentile(similaridade_treino_interna, PERCENTIL_DOMINIO))
print("limiar de dominio (percentil", PERCENTIL_DOMINIO, "do treino):", round(LIMIAR_DOMINIO, 3))

In [ ]:
# Célula 51 — Histograma da similaridade máxima ao treino
figura_dominio = px.histogram(
    x=similaridade_teste, nbins=40,
    labels={"x": "Tanimoto maxima ao treino"},
    title="Similaridade das moleculas de teste ao treino")
figura_dominio.add_vline(x=LIMIAR_DOMINIO, line_dash="dash", line_color="#c0392b",
                         annotation_text="limiar de dominio")
figura_dominio.update_traces(marker_color="#3266ad")
figura_dominio.show()

### 7.2 — O botão do domínio: cobertura × acerto

`PERCENTIL_DOMINIO` é um botão com um trade-off claro. Baixá-lo **alarga** o domínio
(o modelo opina sobre mais moléculas — maior **cobertura**), mas passa a arriscar
palpite em casos mais atípicos (o **acerto** ali pode cair). Em vez de escolher no
escuro, **varremos** o percentil e vemos as duas curvas. A cobertura é a fração de
moléculas de teste dentro do domínio; o acerto é a acurácia da floresta (Seção 5)
**apenas nessas** moléculas. A linha verde marca a nossa escolha (2).

In [ ]:
# Célula 52 — Varredura do percentil do domínio: cobertura × acerto
percentis = [1, 2, 5, 10, 20, 30]
cobertura_por_percentil = []
acerto_por_percentil = []
predito_teste_floresta = modelo_floresta.predict(X_teste)
print("percentil  limiar  cobertura  acerto(dentro)")
for percentil in percentis:
    limiar = float(np.percentile(similaridade_treino_interna, percentil))
    dentro = similaridade_teste >= limiar
    cobertura = float(dentro.mean())
    if int(dentro.sum()) > 0:
        acerto = float((predito_teste_floresta[dentro] == y_teste[dentro]).mean())
    else:
        acerto = float("nan")
    cobertura_por_percentil.append(cobertura)
    acerto_por_percentil.append(acerto)
    print("%8d   %.3f    %.3f      %.3f" % (percentil, limiar, cobertura, acerto))

In [ ]:
# Célula 53 — Plota cobertura × acerto por percentil
figura_botao, eixo_cob = plt.subplots(figsize=(5.4, 3.4))
eixo_cob.plot(percentis, [100 * c for c in cobertura_por_percentil], "o-", color="#3266ad")
eixo_cob.set_xlabel("PERCENTIL_DOMINIO (maior = dominio mais estrito)")
eixo_cob.set_ylabel("cobertura do teste (%)", color="#3266ad")
eixo_cob.tick_params(axis="y", labelcolor="#3266ad")
eixo_ac = eixo_cob.twinx()
eixo_ac.plot(percentis, [100 * a for a in acerto_por_percentil], "s--", color="#c0392b")
eixo_ac.set_ylabel("acerto nas moleculas dentro (%)", color="#c0392b")
eixo_ac.tick_params(axis="y", labelcolor="#c0392b")
eixo_cob.axvline(PERCENTIL_DOMINIO, color="#7e9603", linestyle=":", linewidth=2)
eixo_cob.set_title("Dominio: cobertura x acerto ao variar o percentil")
plt.tight_layout(); plt.show()

Leia a **tendência**, não um ponto: à medida que o percentil sobe (domínio
mais estrito), a cobertura cai e o acerto entre as moléculas de dentro tende a
subir — o modelo passa a opinar só onde está mais seguro. O acerto pode oscilar
porque o grupo "dentro" muda de tamanho e composição a cada passo (poucas moléculas
atípicas pesam muito). A escolha do percentil 2 privilegia **cobertura**: opinar
sobre mais, assumindo o risco em troca — coerente com uma triagem, onde a abstenção
excessiva desperdiça candidatos.

## Seção 8 — O classificador em uso

Até aqui o código foi **procedural** — cada célula mostra os passos, sem funções
escondendo a lógica. Agora, porém, uma função **se justifica**: `classificar` é a
**ferramenta** que queremos reutilizar em qualquer molécula nova, e reutilização é
exatamente para o que servem as funções. É o único caso deste tipo na aula (fora a
classe da rede em PyTorch, que o framework exige).

A função `classificar(smiles)` devolve **FORTE**, **FRACO** ou **INDEFINIDA**,
sempre com o motivo. A ordem importa: primeiro verificamos o **domínio de
aplicabilidade** (Seção 7); só então interpretamos a probabilidade. Uma molécula
fora do domínio é INDEFINIDA por atipicidade, independentemente do que o modelo
"acharia".

In [ ]:
# Célula 54 — A função classificar(smiles), com abstenção
MARGEM_ABSTENCAO = 0.15   # se |proba - 0.5| < margem, o modelo se abstem (INDEFINIDA)

def classificar(smiles):
    """Classifica uma molecula (SMILES) como FORTE, FRACO ou INDEFINIDA, com motivo."""
    molecula = Chem.MolFromSmiles(smiles)
    if molecula is None:
        return {"classe": "INDEFINIDA", "motivo": "SMILES invalido",
                "prob_forte": None, "similaridade": None}

    # 1. dominio de aplicabilidade primeiro
    fp = AllChem.GetMorganFingerprintAsBitVect(molecula, RAIO_MORGAN, nBits=N_BITS)
    similaridade = max(DataStructs.BulkTanimotoSimilarity(fp, fp_treino_rdkit))
    if similaridade < LIMIAR_DOMINIO:
        return {"classe": "INDEFINIDA",
                "motivo": "fora do dominio (Tanimoto max %.2f < %.2f)" % (similaridade, LIMIAR_DOMINIO),
                "prob_forte": None, "similaridade": similaridade}

    # 2. so agora a probabilidade do modelo. Monta o mesmo vetor de features do treino:
    #    os 9 descritores (na ordem das colunas) seguidos dos 2048 bits do fingerprint.
    vetor_desc = np.array([
        Descriptors.MolWt(molecula), Descriptors.MolLogP(molecula),
        Descriptors.TPSA(molecula), Descriptors.NumHDonors(molecula),
        Descriptors.NumHAcceptors(molecula), Descriptors.NumRotatableBonds(molecula),
        Descriptors.NumAromaticRings(molecula), Descriptors.FractionCSP3(molecula),
        molecula.GetNumHeavyAtoms(),
    ], dtype=float)
    vetor_fp = np.zeros((N_BITS,), dtype=float)
    DataStructs.ConvertToNumpyArray(fp, vetor_fp)
    entrada = np.hstack([vetor_desc, vetor_fp]).reshape(1, -1)
    probabilidade_forte = float(modelo_floresta.predict_proba(entrada)[0, 1])

    # 3. abstencao por ambiguidade estatistica
    if abs(probabilidade_forte - 0.5) < MARGEM_ABSTENCAO:
        return {"classe": "INDEFINIDA",
                "motivo": "ambiguo (probabilidade FORTE = %.2f)" % probabilidade_forte,
                "prob_forte": probabilidade_forte, "similaridade": similaridade}
    if probabilidade_forte >= 0.5:
        return {"classe": "FORTE", "motivo": "probabilidade FORTE = %.2f" % probabilidade_forte,
                "prob_forte": probabilidade_forte, "similaridade": similaridade}
    return {"classe": "FRACO", "motivo": "probabilidade FORTE = %.2f" % probabilidade_forte,
            "prob_forte": probabilidade_forte, "similaridade": similaridade}

# teste rapido com uma molecula do proprio conjunto
exemplo_smiles = agregados.loc[idx_teste_esq[0], "canonical_smiles"]
print(exemplo_smiles, "->", classificar(exemplo_smiles))

### 8.1 — Galeria de teste

Testamos com uma grade de moléculas desenhadas pelo RDKit, com a resposta do
modelo como legenda: inibidores conhecidos da acetilcolinesterase (donepezila,
tacrina, galantamina), fármacos sem relação com o alvo (aspirina, paracetamol) e
moléculas cotidianas (cafeína, etanol).

Espere respostas de três naturezas, e **leia os motivos impressos**, não só as
classes: moléculas muito diferentes de tudo que o modelo viu (como o etanol) caem
em INDEFINIDA por atipicidade; um inibidor conhecido de esqueleto familiar tende a
FORTE; e há um caso instrutivo — uma molécula "cotidiana" pode, ainda assim,
**estar dentro do domínio** se sua estrutura se parece com algo do treino, e então
receber uma resposta confiante. Não presuma o resultado: observe o que o modelo de
fato responde e por quê.

In [ ]:
# Célula 55 — Galeria de moléculas de teste
galeria = [
    ("donepezila",  "O=C1CC2(CCN(Cc3ccccc3)CC2)Cc2cc(OC)c(OC)cc21"),
    ("tacrina",     "Nc1c2c(nc3ccccc13)CCCC2"),
    ("galantamina", "CN1CCC23C=CC(O)CC2Oc2c(OC)ccc(c23)C1"),
    ("aspirina",    "CC(=O)Oc1ccccc1C(=O)O"),
    ("paracetamol", "CC(=O)Nc1ccc(O)cc1"),
    ("cafeina",     "Cn1cnc2c1c(=O)n(C)c(=O)n2C"),
    ("etanol",      "CCO"),
]
moleculas_galeria = []
legendas_galeria = []
for nome, smiles in galeria:
    resposta = classificar(smiles)
    moleculas_galeria.append(Chem.MolFromSmiles(smiles))
    legendas_galeria.append(nome + ": " + resposta["classe"])
    print(f"{nome:14s} -> {resposta['classe']:8s} ({resposta['motivo']})")

Draw.MolsToGridImage(moleculas_galeria, legends=legendas_galeria,
                     molsPerRow=4, subImgSize=(230, 180))

### 8.2 — Triagem virtual: procurar inibidores entre fármacos aprovados

Agora o pagamento de verdade da aula. Pegamos uma **biblioteca real** de fármacos
aprovados no mundo (o conjunto `world` do ZINC15, cerca de 5,9 mil moléculas) e
passamos **cada uma** pelo `classificar`. A pergunta prática: **existe, entre remédios que já são
usados para outras coisas, algum candidato a inibidor da acetilcolinesterase?** É
exatamente o raciocínio do **reposicionamento de fármacos** — e é para isso que a
`classificar` foi feita.

Carregamos o arquivo direto do repositório (`data/world.csv`), com as colunas
`zinc_id` (identificador) e `smiles` (estrutura).

In [ ]:
# Célula 56 — Triagem virtual sobre fármacos aprovados (world)
URL_WORLD = ("https://raw.githubusercontent.com/monteirotorres/ml/"
             "main/data/world.csv")
try:
    tabela_world = pd.read_csv(URL_WORLD)
    print("biblioteca lida da URL:", len(tabela_world), "farmacos aprovados")
except Exception as erro:
    print("URL indisponivel (", type(erro).__name__, "); tentando arquivo local")
    tabela_world = pd.read_csv("world.csv")
    print("biblioteca lida do arquivo local:", len(tabela_world))
tabela_world.head()

Rodamos o classificador em toda a biblioteca. Como a imensa maioria dos
fármacos aprovados **não** tem nada a ver com a acetilcolinesterase, o esperado é
um mar de FRACO e INDEFINIDA e **poucos** FORTE — e são justamente esses poucos que
interessam. Guardamos a probabilidade de FORTE para ordenar os candidatos depois.

In [ ]:
# Célula 57 — Passa cada fármaco pelo classificador
# passa cada farmaco pela ferramenta da Secao 8 (reuso, nao reimplementacao)
linhas_triagem = []
for posicao in range(len(tabela_world)):
    smiles = tabela_world.iloc[posicao]["smiles"]
    zinc_id = tabela_world.iloc[posicao]["zinc_id"]
    resposta = classificar(smiles)
    linhas_triagem.append({
        "zinc_id": zinc_id, "smiles": smiles,
        "classe": resposta["classe"], "prob_forte": resposta["prob_forte"],
    })
triagem = pd.DataFrame(linhas_triagem)

print("distribuicao das", len(triagem), "moleculas aprovadas:")
contagem_triagem = triagem["classe"].value_counts()
for nome_classe in ["FORTE", "FRACO", "INDEFINIDA"]:
    n = int(contagem_triagem.get(nome_classe, 0))
    print(f"  {nome_classe:11s}: {n:4d}  ({round(100 * n / len(triagem), 1)}%)")

### 8.2a — Sanidade: recuperamos os inibidores já conhecidos?

Antes de acreditar num candidato novo, uma pergunta de **controle**: dos fármacos
desta biblioteca que **já têm medida de IC50 no ChEMBL** contra a acetilcolinesterase
(ou seja, cuja potência real conhecemos), o modelo acerta? Achamos essa interseção
casando o **esqueleto InChIKey** (robusto a sal/estereo/protonação) e comparamos a
classe do modelo com o rótulo verdadeiro.

**Ressalva honesta e importante:** alguns desses conhecidos estavam no **treino** do
modelo — acertá-los ali é em parte memorização, não generalização. Por isso separamos
os que estavam no treino dos que **não** estavam (o teste honesto), e olhamos os dois
grupos.

In [ ]:
# Célula 58 — Sanidade: recuperamos os inibidores já conhecidos?
# mapa: esqueleto-InChIKey -> (rotulo verdadeiro, estava no treino do modelo?)
def esqueleto_inchikey(molecula):
    return Chem.MolToInchiKey(molecula).split("-")[0]

conjunto_treino = set(idx_treino_esq)
verdade_por_esqueleto = {}
for indice in agregados.index:
    chave = esqueleto_inchikey(agregados.loc[indice, "molecula"])
    verdade_por_esqueleto[chave] = {
        "rotulo": int(agregados.loc[indice, "rotulo"]),
        "no_treino": indice in conjunto_treino,
    }

# quais farmacos da biblioteca tambem estao no ChEMBL AChE (esqueleto conhecido)
linhas_conhecidas = []
for posicao in range(len(triagem)):
    smiles = triagem.iloc[posicao]["smiles"]
    molecula = Chem.MolFromSmiles(smiles)
    if molecula is None:
        continue
    chave = esqueleto_inchikey(molecula)
    if chave in verdade_por_esqueleto:
        verdade = verdade_por_esqueleto[chave]
        linhas_conhecidas.append({
            "zinc_id": triagem.iloc[posicao]["zinc_id"],
            "classe_modelo": triagem.iloc[posicao]["classe"],
            "verdade": "FORTE" if verdade["rotulo"] == 1 else "FRACO",
            "no_treino": verdade["no_treino"],
        })
conhecidas = pd.DataFrame(linhas_conhecidas)
print("farmacos aprovados que TAMBEM tem medida no ChEMBL AChE:", len(conhecidas))

# entre os que a VERDADE diz FORTE: quantos o modelo recupera como FORTE?
verdadeiros_fortes = conhecidas[conhecidas["verdade"] == "FORTE"]
print("\ndestes, com potencia real FORTE:", len(verdadeiros_fortes))
for rotulo_grupo, grupo in [("no treino (memoria)", verdadeiros_fortes[verdadeiros_fortes["no_treino"]]),
                            ("fora do treino (teste honesto)", verdadeiros_fortes[~verdadeiros_fortes["no_treino"]])]:
    if len(grupo) == 0:
        print(f"  {rotulo_grupo}: nenhum")
        continue
    n_forte = int((grupo["classe_modelo"] == "FORTE").sum())
    n_fraco = int((grupo["classe_modelo"] == "FRACO").sum())
    n_indef = int((grupo["classe_modelo"] == "INDEFINIDA").sum())
    print(f"  {rotulo_grupo}: {len(grupo)} conhecidos -> "
          f"FORTE {n_forte} | FRACO {n_fraco} (falsos negativos) | INDEFINIDA {n_indef}")

### 8.2b — Os candidatos novos

Agora o que a triagem tem de mais interessante: fármacos classificados **FORTE**
cujo esqueleto **não** aparece no ChEMBL AChE — ou seja, moléculas para as quais o
modelo aposta em atividade **sem** ter visto nada igual medido contra o alvo. Cada
uma é uma **hipótese de reposicionamento** a levar para a bancada, não uma verdade:
o modelo dá a pista, o experimento decide. Ordenamos pela probabilidade de FORTE e
desenhamos os primeiros.

In [ ]:
# Célula 59 — Os candidatos novos da triagem
esqueletos_conhecidos = set(verdade_por_esqueleto.keys())

candidatas_novas = []
for posicao in range(len(triagem)):
    if triagem.iloc[posicao]["classe"] != "FORTE":
        continue
    smiles = triagem.iloc[posicao]["smiles"]
    molecula = Chem.MolFromSmiles(smiles)
    if molecula is None:
        continue
    if esqueleto_inchikey(molecula) in esqueletos_conhecidos:
        continue                                   # ja conhecida, nao e novidade
    candidatas_novas.append((triagem.iloc[posicao]["zinc_id"], smiles,
                             float(triagem.iloc[posicao]["prob_forte"])))

candidatas_novas.sort(key=lambda t: -t[2])         # maior probabilidade primeiro
print("candidatas NOVAS classificadas FORTE (esqueleto inedito):", len(candidatas_novas))

n_mostrar = min(8, len(candidatas_novas))
moleculas_cand = []
legendas_cand = []
for zinc_id, smiles, prob in candidatas_novas[:n_mostrar]:
    moleculas_cand.append(Chem.MolFromSmiles(smiles))
    legendas_cand.append(zinc_id + "  p=" + str(round(prob, 2)))
    print(f"  {zinc_id}  prob_forte={round(prob, 2)}  {smiles}")

if n_mostrar > 0:
    imagem_candidatas = Draw.MolsToGridImage(moleculas_cand, legends=legendas_cand,
                                             molsPerRow=4, subImgSize=(230, 180))
else:
    imagem_candidatas = None
imagem_candidatas

## Seção 9 — Persistência

Salvamos tudo que a função `classificar` precisa para rodar amanhã sem refazer o
treino: o modelo, os fingerprints do treino (para o domínio), os limiares e um
**dicionário de versões**. As versões importam porque um modelo salvo com uma versão
de scikit-learn ou RDKit pode não recarregar corretamente em outra — sem registrar
isso, um modelo "que funcionava" vira irreproduzível.

In [ ]:
# Célula 60 — Persistência do modelo
pacote = {
    "modelo_floresta": modelo_floresta,
    "fp_treino_rdkit": fp_treino_rdkit,
    "limiar_dominio": LIMIAR_DOMINIO,
    "limiar_potencia": LIMIAR_POTENCIA,
    "margem_abstencao": MARGEM_ABSTENCAO,
    "raio_morgan": RAIO_MORGAN,
    "n_bits": N_BITS,
    "nomes_descritores": list(tabela_descritores.columns),
    "versoes": {
        "numpy": np.__version__, "pandas": pd.__version__,
        "sklearn": sklearn.__version__, "torch": torch.__version__,
    },
}
joblib.dump(pacote, "classificador_ache.joblib")
print("salvo: classificador_ache.joblib")
print("versoes registradas:", pacote["versoes"])